[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jhavierc/NLP_taller_1/blob/main/notebook_taller_3_v2.ipynb)

> **Nota sobre Google Colab:** igual que en taller 1, este cuaderno está pensado para 2 GPU Tesla T4 (Kaggle). En Colab gratuito (1 GPU) cambia `requested_gpus=1` en el PASO 02 antes de ejecutar.

# Taller 3 — BETO y RoBERTa: encoder congelado frente a ajuste completo

**Carlos Javier Cepeda, David Salamanca, Jose Milciades Ordoñez**

## Objetivo
Resolvemos el mismo problema del taller 1 — reconocimiento de entidades clínicas (síntomas, enfermedades, procedimientos, fármacos y proteínas) sobre el corpus SPACCC en español — pero reemplazamos la arquitectura Bi-LSTM entrenada desde cero por transformers preentrenados, haciendo fine-tuning con la librería transformers de Hugging Face.

## Qué cambia y que no respecto a taller 1 y 2
- Modelo: en vez de una Bi-LSTM con embeddings entrenados desde cero, usamos dos transformers preentrenados en español:
dccuchile/bert-base-spanish-wwm-cased (BETO, BERT genérico, dominio general).
PlanTL-GOB-ES/bsc-bio-ehr-es (RoBERTa preentrenado en textos biomédicos y clínicos en español; técnicamente es RoBERTa, no BERT literal, pero lo documentamos igual porque es la mejor opción de dominio disponible en español).
- Tokenización: utilizamos spaCy únicamente para establecer las palabras de referencia compartidas por ambos modelos, sin POS tagging. Luego, el tokenizador de cada Transformer las divide en subtokens. Supervisamos solo el primer subtoken de cada palabra.
- Entrenamiento: utilizamos Trainer de Hugging Face para gestionar el entrenamiento, la evaluación, la parada temprana y la selección del mejor checkpoint. En Kaggle, con dos GPU CUDA visibles y ejecución en un solo proceso, Trainer gestiona el paralelismo sin implementar DataParallel manualmente.
- Variantes de entrenamiento: comparamos frozen, donde el encoder permanece congelado y solo se entrena el cabezal de clasificación, y fine_tuned, donde se actualizan tanto el encoder como el cabezal. En esta prueba ejecutamos 2 modelos × 2 variantes × 1 semilla —123— = 4 entrenamientos.
- Elementos compartidos: conservamos el corpus SPACCC, la partición fija por documento —semilla 42, con 600 documentos de entrenamiento, 150 de validación y 250 de prueba—, la política de resolución de solapamientos flat longest y las cinco categorías clínicas: CHEMICAL, DISEASE, PROCEDURE, PROTEIN y SYMPTOM, representadas mediante 11 etiquetas BIO.
- Comparabilidad: BETO y RoBERTa comparten las mismas entidades de referencia y evaluación por documento en la v2.
Las comparaciones con talleres anteriores deben considerar las diferencias de tokenización y evaluación; compartir corpus y partición no basta para garantizar equivalencia completa.



### PASO 01 — Dependencias

Instalamos Transformers, datasets, accelerate. El tokenizador de `es_core_news_sm` define las mismas palabras de referencia para BETO y RoBERTa; se desactivan sus componentes de análisis lingüístico. No se utilizan etiquetas POS. La descarga requiere Internet.


In [9]:
# PASO 01 — Dependencias (Internet activado; ejecutar una vez por sesion)
import os, sys, subprocess, importlib.util

def pip_install(*args):
    """Corre pip capturando su salida: si falla, imprime el error real de pip en
    vez de un CalledProcessError vacio (necesario porque -q oculta el traceback de
    fondo que explica CUAL paquete fallo y por que)."""
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-4000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f'pip install fallo (codigo {result.returncode}): {args}')

# Kaggle a veces trae un pip desactualizado que no sabe generar metadatos para
# paquetes "legacy" (setup.py sin pyproject.toml); actualizarlo primero evita el
# error "python setup.py egg_info did not run successfully".
pip_install('--upgrade', 'pip', 'setuptools', 'wheel')

packages = ['transformers==5.6.0', 'datasets>=5,<6', 'accelerate>=1.1',
            'pandas>=2,<3', 'pyarrow>=14', 'requests>=2.31', 'tqdm>=4.66', 'spacy>=3.8,<4']
if importlib.util.find_spec('torch') is None:
    packages.append('torch>=2.4,<3')
pip_install(*packages)
if importlib.util.find_spec('es_core_news_sm') is None:
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'es_core_news_sm'])
if 'transformers' in sys.modules and sys.modules['transformers'].__version__ != '5.6.0':
    raise RuntimeError('Reinicia la sesión de Kaggle: Transformers se actualizó, pero la versión anterior sigue en memoria.')
print('PASO 01 OK. Dependencias instaladas. Continua con la PASO 02.')


PASO 01 OK. Dependencias instaladas. Continua con la PASO 02.


### PASO 02 — Cuatro escenarios con semilla 123

Ambos modelos ejecutan `frozen` con tasa `1e-3` y `fine_tuned` con tasa `2e-5`. Longitud máxima 512, lote 8 por GPU, hasta 8 épocas y paciencia 2. La comprobación técnica usa 20 pasos sobre un lote pequeño del entrenamiento. La partición conserva semilla 42; el test no participa en selección. Se puede ampliar la lista de semillas después de revisar esta prueba.


In [10]:
# PASO 02 - Prueba inicial comparable de dos modelos
import os, json, time, random, math, hashlib, platform, traceback
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import requests, torch
from torch import nn

CFG = dict(
    split_seed=42, training_seeds=[123], fraction=1.0, validation_fraction=0.20,
    max_length=512, overlap_subtokens=64, batch_size=8, patience=2, weight_decay=0.01, warmup_steps=0.1,
    learning_rate={'frozen': 1e-3, 'fine_tuned': 2e-5},
    epochs={'frozen': 8, 'fine_tuned': 8},
    class_weights=False,
    checkpoints={'beto': 'dccuchile/bert-base-spanish-wwm-cased',
                 'clinical': 'PlanTL-GOB-ES/bsc-bio-ehr-es'},
    variants=['frozen', 'fine_tuned'], smoke_steps=20,
    run_final_test=False, final_test_checkpoint=None, final_test_variant=None, final_test_seed=None,
    requested_gpus=2, allow_cpu_for_debug=False,
)
assert 0 < CFG['fraction'] <= 1
assert 0 < CFG['validation_fraction'] < 1, 'Usa validation_fraction=0.20 para reservar el 20 %.'
assert isinstance(CFG['split_seed'], int) and 0 <= CFG['split_seed'] < 2**32
assert (isinstance(CFG['training_seeds'], list) and len(CFG['training_seeds']) >= 1
        and all(type(s) is int and 0 <= s < 2**32 for s in CFG['training_seeds'])
        and len(set(CFG['training_seeds'])) == len(CFG['training_seeds'])), 'Usa una o más semillas enteras distintas.'
random.seed(CFG['split_seed']); np.random.seed(CFG['split_seed']); torch.manual_seed(CFG['split_seed'])
assert CFG['variants'] and set(CFG['variants']) <= {'frozen', 'fine_tuned'}
assert len(CFG['variants']) == len(set(CFG['variants']))
assert 0 <= CFG['overlap_subtokens'] < CFG['max_length'] - 2
assert CFG['requested_gpus'] in (1, 2), 'requested_gpus debe ser 1 o 2.'
available_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if available_gpus == 0 and CFG['allow_cpu_for_debug']:
    device = torch.device('cpu')
    print('AVISO: modo de depuracion CPU explicito; NO valida CUDA ni las dos T4.')
elif available_gpus < CFG['requested_gpus']:
    raise RuntimeError(f'PASO 02: se requieren {CFG["requested_gpus"]} GPU CUDA y se detectaron '
                       f'{available_gpus}. En Kaggle selecciona GPU T4 x2 y reinicia la sesion. '
                       'Para usar deliberadamente una sola GPU cambia requested_gpus=1.')
else:
    device = torch.device('cuda:0')
    torch.cuda.set_device(device)
    torch.cuda.manual_seed_all(CFG['split_seed'])
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
WORK = ROOT / 'spaccc_transformers'
WORK.mkdir(parents=True, exist_ok=True)
RUN = WORK / (time.strftime('%Y%m%d_%H%M%S') + '_' + str(time.time_ns())[-6:])
RUN.mkdir()
CACHE = WORK / 'cache'; CACHE.mkdir(exist_ok=True)
LABELS = ['CHEMICAL', 'DISEASE', 'PROCEDURE', 'PROTEIN', 'SYMPTOM']
TAGS = ['O'] + [f'{prefix}-{label}' for label in LABELS for prefix in ('B', 'I')]
tag_to_ix = {tag: i for i, tag in enumerate(TAGS)}
id2label = {i: tag for i, tag in enumerate(TAGS)}
label2id = {tag: i for i, tag in enumerate(TAGS)}
IGNORE = -100  # Convencion de Hugging Face: posiciones que no se evaluan (padding, subwords de continuacion, tokens especiales).
START = time.perf_counter()
REPORT = {'config': CFG.copy(), 'run_dir': str(RUN), 'device': str(device),
          'gpu': {'available_count': available_gpus, 'requested': CFG['requested_gpus'],
                  'devices': [{'id': i, 'name': torch.cuda.get_device_name(i),
                               'total_memory_gb': torch.cuda.get_device_properties(i).total_memory / 2**30}
                              for i in range(available_gpus)]},
          'cpu': {'logical_cores': os.cpu_count(), 'machine': platform.machine()},
          'versions': {'python': platform.python_version(), 'torch': str(torch.__version__)}}
(RUN / 'config.json').write_text(json.dumps(REPORT, indent=2), encoding='utf-8')
from html import escape
from IPython.display import Markdown, display

def mostrar_tabla_configuracion(titulo, datos):
    """Muestra los campos anidados como filas de una tabla Markdown."""
    def filas(campos, prefijo=''):
        for clave, valor in campos.items():
            nombre = f'{prefijo}.{clave}' if prefijo else clave
            if isinstance(valor, dict):
                yield from filas(valor, nombre)
            else:
                if isinstance(valor, list):
                    valor = ', '.join(map(str, valor))
                elif valor is None:
                    valor = 'Sin definir'
                elif isinstance(valor, bool):
                    valor = 'Sí' if valor else 'No'
                yield nombre, valor

    def texto(valor):
        return escape(str(valor)).replace('|', '&#124;').replace('\n', '<br>')

    lineas = [f'#### {titulo}', '', '| Parámetro | Valor |', '| --- | --- |']
    lineas.extend(f'| {texto(clave)} | {texto(valor)} |' for clave, valor in filas(datos))
    display(Markdown('\n'.join(lineas)))

display(Markdown('### PASO 02 OK'))
mostrar_tabla_configuracion('Configuración', REPORT['config'])
mostrar_tabla_configuracion('Entorno de ejecución', {
    'run_dir': REPORT['run_dir'], 'device': REPORT['device'],
    'gpu_disponibles': REPORT['gpu']['available_count'],
    'gpu_solicitadas': REPORT['gpu']['requested'],
    'cpu': REPORT['cpu'], 'versiones': REPORT['versions'],
})
for gpu in REPORT['gpu']['devices']:
    mostrar_tabla_configuracion(f"GPU {gpu['id']}", gpu)
# Tablas Markdown sin dependencia adicional de tabulate.
def mostrar_tabla(titulo, columnas, filas):
    def texto(x):
        if x is None:
            return 'No estimable'
        return escape(str(x)).replace('|', '&#124;').replace('\n', '<br>')
    lineas = [f'#### {titulo}', '', '| ' + ' | '.join(map(texto, columnas)) + ' |',
              '| ' + ' | '.join(['---'] * len(columnas)) + ' |']
    lineas.extend('| ' + ' | '.join(map(texto, fila)) + ' |' for fila in filas)
    display(Markdown('\n'.join(lineas)))


### PASO 02 OK

#### Configuración

| Parámetro | Valor |
| --- | --- |
| split_seed | 42 |
| training_seeds | 123 |
| fraction | 1.0 |
| validation_fraction | 0.2 |
| max_length | 512 |
| overlap_subtokens | 64 |
| batch_size | 8 |
| patience | 2 |
| weight_decay | 0.01 |
| warmup_steps | 0.1 |
| learning_rate.frozen | 0.001 |
| learning_rate.fine_tuned | 2e-05 |
| epochs.frozen | 8 |
| epochs.fine_tuned | 8 |
| class_weights | No |
| checkpoints.beto | dccuchile/bert-base-spanish-wwm-cased |
| checkpoints.clinical | PlanTL-GOB-ES/bsc-bio-ehr-es |
| variants | frozen, fine_tuned |
| smoke_steps | 20 |
| run_final_test | No |
| final_test_checkpoint | Sin definir |
| final_test_variant | Sin definir |
| final_test_seed | Sin definir |
| requested_gpus | 2 |
| allow_cpu_for_debug | No |

#### Entorno de ejecución

| Parámetro | Valor |
| --- | --- |
| run_dir | /kaggle/working/spaccc_transformers/20260926_235331_750165 |
| device | cuda:0 |
| gpu_disponibles | 2 |
| gpu_solicitadas | 2 |
| cpu.logical_cores | 4 |
| cpu.machine | x86_64 |
| versiones.python | 3.12.13 |
| versiones.torch | 2.10.0+cu128 |

#### GPU 0

| Parámetro | Valor |
| --- | --- |
| id | 0 |
| name | Tesla T4 |
| total_memory_gb | 14.56219482421875 |

#### GPU 1

| Parámetro | Valor |
| --- | --- |
| id | 1 |
| name | Tesla T4 |
| total_memory_gb | 14.56219482421875 |

### PASO 03 - Descarga y auditoria de los datos

Reutilizamos exactamente el mismo procedimiento de taller 1: descargamos y dejamos en cache las anotaciones NER, los documentos completos y el conjunto de prueba desde Hugging Face (mismo corpus SPACCC). Comprobamos columnas, categorias, duplicacion de textos entre particiones y correspondencia entre cada offset y el texto original. El analisis exploratorio detallado (distribucion de clases, longitud de entidades, densidad por documento) ya se hizo en taller 1 y no se repite aqui; esta PASO solo audita que la descarga sea consistente antes de tokenizar con los transformers.

In [11]:
# PASO 03 - Descarga, textos completos y auditoria de todas las anotaciones (igual que taller 1)
def download_parquet(repo, split, name):
    """Descarga el parquet `split` del dataset `repo` (Hugging Face) y lo guarda como
    `name` dentro de la carpeta de cache. Si el archivo ya existe en cache lo reutiliza
    sin volver a descargarlo. Reintenta la descarga hasta 3 veces y valida que el
    parquet descargado se pueda leer antes de reemplazar la cache."""
    path = CACHE / name
    if not path.exists():
        url = f'https://huggingface.co/datasets/{repo}/resolve/main/data/{split}-00000-of-00001.parquet'
        print('Descargando:', name, flush=True)
        error = None
        for attempt in range(3):
            try:
                response = requests.get(url, timeout=(15, 120))
                response.raise_for_status()
                tmp = path.with_suffix('.tmp')
                tmp.write_bytes(response.content)
                pd.read_parquet(tmp)  # No guardar una respuesta invalida como cache.
                tmp.replace(path)
                break
            except Exception as exc:
                error = exc
                print(f'Intento {attempt + 1}/3: {type(exc).__name__}: {exc}', flush=True)
        if not path.exists():
            raise RuntimeError('PASO 03: revisa Internet de Kaggle y comparte este error.') from error
    return pd.read_parquet(path)

t0 = time.perf_counter()
df_train = download_parquet('IEETA/SPACCC-Spanish-NER', 'train', 'annotations_train.parquet')
df_test = download_parquet('IEETA/SPACCC-Spanish-NER', 'test', 'annotations_test.parquet')
df_docs = download_parquet('IEETA/SPACCC-documents', 'train', 'documents.parquet')
def document_id(value):
    """Obtiene el identificador de un documento a partir de su nombre de archivo."""
    return Path(str(value)).stem.removeprefix('es-')
required = {'filename', 'label', 'start_span', 'end_span', 'text'}
for frame in (df_train, df_test):
    assert required <= set(frame.columns), 'Columnas de anotacion inesperadas.'
    assert not frame[list(required)].isnull().any().any(), 'Anotaciones con valores nulos.'
    frame['doc_id'] = frame.filename.map(document_id)
    assert set(frame.label) <= set(LABELS), 'Categorias nuevas: revisar TAGS.'
df_docs['doc_id'] = df_docs.filename.map(document_id)
assert not df_docs.doc_id.duplicated().any(), 'Identificadores de documento duplicados.'
assert df_docs.document.map(lambda x: isinstance(x, str) and bool(x)).all()
texts = dict(zip(df_docs.doc_id, df_docs.document))
official_train_ids = sorted(df_train.doc_id.unique())
official_test_ids = sorted(df_test.doc_id.unique())
assert not set(official_train_ids) & set(official_test_ids), 'Fuga: documentos en train y test.'
assert (set(official_train_ids) | set(official_test_ids)) <= texts.keys(), 'Faltan documentos completos.'
bad_offsets, quote_only_differences = [], []
for split, frame in [('train', df_train), ('test', df_test)]:
    for row in frame.itertuples(index=False):
        a, b = int(row.start_span), int(row.end_span)
        actual = texts[row.doc_id][a:b]
        detail = {'split': split, 'doc_id': row.doc_id, 'start': a, 'end': b}
        if not (0 <= a < b <= len(texts[row.doc_id])):
            bad_offsets.append(detail)
        elif actual != row.text:
            if actual.replace(chr(34), '') == row.text.replace(chr(34), ''):
                quote_only_differences.append(detail)
            else:
                bad_offsets.append(detail)
assert not bad_offsets, f'Offsets incompatibles: {len(bad_offsets)}. Ejemplos: {bad_offsets[:5]}'
fingerprint = lambda s: hashlib.sha256(' '.join(s.split()).encode()).hexdigest()
train_hashes = {fingerprint(texts[k]) for k in official_train_ids}
test_hashes = {fingerprint(texts[k]) for k in official_test_ids}
assert not train_hashes & test_hashes, 'Textos duplicados entre train y test: revisar particion.'
REPORT['data'] = {'train_documents': len(official_train_ids), 'test_documents': len(official_test_ids),
                  'train_annotations': len(df_train), 'test_annotations': len(df_test),
                  'offset_errors': len(bad_offsets), 'quote_only_differences': quote_only_differences,
                  'download_audit_seconds': time.perf_counter() - t0,
                  'sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                             for p in CACHE.glob('*.parquet')}}
display(Markdown('### PASO 03 OK'))
mostrar_tabla_configuracion('Resumen de descarga y auditoría', {
    'Documentos de entrenamiento': REPORT['data']['train_documents'],
    'Documentos de prueba': REPORT['data']['test_documents'],
    'Anotaciones de entrenamiento': REPORT['data']['train_annotations'],
    'Anotaciones de prueba': REPORT['data']['test_annotations'],
    'Errores de offsets': REPORT['data']['offset_errors'],
    'Diferencias solo por comillas': len(quote_only_differences),
    'Tiempo de descarga y auditoría (s)': round(REPORT['data']['download_audit_seconds'], 3),
})
mostrar_tabla_configuracion('Huellas SHA-256 de los archivos', REPORT['data']['sha256'])
if quote_only_differences:
    lineas_diferencias = [
        '#### Detalle de diferencias solo por comillas', '',
        '| Partición | Documento | Inicio | Fin |', '| --- | --- | ---: | ---: |',
    ]
    for diferencia in quote_only_differences:
        valores = [escape(str(diferencia[campo])).replace('|', '&#124;').replace('\n', '<br>')
                   for campo in ('split', 'doc_id', 'start', 'end')]
        lineas_diferencias.append('| ' + ' | '.join(valores) + ' |')
    display(Markdown('\n'.join(lineas_diferencias)))

Descargando: annotations_train.parquet
Descargando: annotations_test.parquet
Descargando: documents.parquet


### PASO 03 OK

#### Resumen de descarga y auditoría

| Parámetro | Valor |
| --- | --- |
| Documentos de entrenamiento | 750 |
| Documentos de prueba | 250 |
| Anotaciones de entrenamiento | 33757 |
| Anotaciones de prueba | 11239 |
| Errores de offsets | 0 |
| Diferencias solo por comillas | 2 |
| Tiempo de descarga y auditoría (s) | 2.23 |

#### Huellas SHA-256 de los archivos

| Parámetro | Valor |
| --- | --- |
| documents.parquet | d866b787254d0ebdc4e7453ea3fdce25edfb1fc058b536404d1dcf15e5a40f24 |
| annotations_train.parquet | 76c692d7cc49d35d030aeb5c27523862ebd7585bdcf31e596f2f7d6ebe37c033 |
| annotations_test.parquet | 53bf36209410f657f773cfeb804b08487ab1e8d23e3af9e972cc3927140e1413 |

#### Detalle de diferencias solo por comillas

| Partición | Documento | Inicio | Fin |
| --- | --- | ---: | ---: |
| test | S0004-06142008000700003-1 | 1560 | 1647 |
| test | S1134-80462005000100007-1 | 127 | 145 |

### Inspección de la carga

Revisamos dimensiones y una muestra de anotaciones de entrenamiento. Los conteos por categoría describen menciones anotadas; todavía no incluyen la etiqueta `O`, las etiquetas BIO ni las exclusiones por solapamiento, que se construirán más adelante.

In [12]:
overview = pd.DataFrame([
    {'recurso': 'Anotaciones train', 'filas': len(df_train), 'documentos': len(official_train_ids)},
    {'recurso': 'Anotaciones test', 'filas': len(df_test), 'documentos': len(official_test_ids)},
    {'recurso': 'Textos completos', 'filas': len(df_docs), 'documentos': df_docs.doc_id.nunique()},
])
display(overview)
display(df_train[['doc_id', 'text', 'label', 'start_span', 'end_span']].head())
display(df_train.label.value_counts().rename_axis('categoria').reset_index(name='anotaciones_train'))
# Catálogo completo; la asignación BIO a tokens se realiza en el PASO 05.
lineas_bio = [
    '#### Todas las etiquetas BIO', '',
    '| ID | Etiqueta BIO | Categoría | Significado |',
    '| ---: | --- | --- | --- |',
]
for tag in TAGS:
    if tag == 'O':
        categoria, significado = 'Sin entidad', 'Token fuera de una entidad'
    else:
        prefijo, categoria = tag.split('-', 1)
        significado = 'Inicio de entidad' if prefijo == 'B' else 'Continuación de entidad'
    lineas_bio.append(f'| {tag_to_ix[tag]} | {tag} | {categoria} | {significado} |')
display(Markdown('\n'.join(lineas_bio)))
display(Markdown(
    'Las anotaciones cargadas contienen categorías y offsets. '
    'Las etiquetas BIO se asignan a los tokens en el **PASO 05**. '
    'El valor `IGNORE = -100` excluye posiciones de la evaluación y no es una etiqueta BIO.'
))

overview.to_csv(WORK / 'data_overview.csv', index=False)


,recurso,filas,documentos
0,Anotaciones train,33757,750
1,Anotaciones test,11239,250
2,Textos completos,1000,1000


,doc_id,text,label,start_span,end_span
0,S0004-06142005000500011-1,alergias medicamentosas,DISEASE,50,73
1,S0004-06142005000500011-1,fracturas vertebrales y costales,DISEASE,158,190
2,S0004-06142005000500011-1,intervenido de enfermedad de Dupuytren en mano...,PROCEDURE,192,278
3,S0004-06142005000500011-1,enfermedad de Dupuytren,DISEASE,207,230
4,S0004-06142005000500011-1,Diabetes Mellitus tipo II,DISEASE,280,305


,categoria,anotaciones_train
0,PROCEDURE,11065
1,SYMPTOM,9091
2,DISEASE,8065
3,CHEMICAL,3283
4,PROTEIN,2253


#### Todas las etiquetas BIO

| ID | Etiqueta BIO | Categoría | Significado |
| ---: | --- | --- | --- |
| 0 | O | Sin entidad | Token fuera de una entidad |
| 1 | B-CHEMICAL | CHEMICAL | Inicio de entidad |
| 2 | I-CHEMICAL | CHEMICAL | Continuación de entidad |
| 3 | B-DISEASE | DISEASE | Inicio de entidad |
| 4 | I-DISEASE | DISEASE | Continuación de entidad |
| 5 | B-PROCEDURE | PROCEDURE | Inicio de entidad |
| 6 | I-PROCEDURE | PROCEDURE | Continuación de entidad |
| 7 | B-PROTEIN | PROTEIN | Inicio de entidad |
| 8 | I-PROTEIN | PROTEIN | Continuación de entidad |
| 9 | B-SYMPTOM | SYMPTOM | Inicio de entidad |
| 10 | I-SYMPTOM | SYMPTOM | Continuación de entidad |

Las anotaciones cargadas contienen categorías y offsets. Las etiquetas BIO se asignan a los tokens en el **PASO 05**. El valor `IGNORE = -100` excluye posiciones de la evaluación y no es una etiqueta BIO.

### Interpretación de la inspección de la carga

La inspección de la carga muestra que el corpus contiene **1.000 documentos clínicos**, distribuidos en 750 documentos de entrenamiento con 33.757 anotaciones y 250 documentos de prueba con 11.239 anotaciones. Ambos conjuntos presentan aproximadamente 45 anotaciones por documento. Los 750 documentos corresponden a la partición original de entrenamiento, que posteriormente se divide en 600 documentos de entrenamiento y 150 de validación.

La distribución de las anotaciones de entrenamiento presenta un **desbalance entre categorías**:

| Categoría | Anotaciones | Porcentaje |
|---|---:|---:|
| PROCEDURE | 11.065 | 32,78 % |
| SYMPTOM | 9.091 | 26,93 % |
| DISEASE | 8.065 | 23,89 % |
| CHEMICAL | 3.283 | 9,73 % |
| PROTEIN | 2.253 | 6,67 % |
| **Total** | **33.757** | **100 %** |

Los procedimientos, síntomas y enfermedades concentran aproximadamente el **83,60 %** de las anotaciones. En cambio, las categorías CHEMICAL y PROTEIN tienen menor representación, lo que puede dificultar su aprendizaje. Esta distribución respalda el uso de ponderación de clases y la evaluación por categoría, ya que una métrica global puede ocultar un desempeño bajo en las clases menos frecuentes.

Cada anotación identifica el documento, el fragmento de texto, su categoría y sus posiciones de inicio y fin. Los ejemplos muestran entidades de varias palabras y anotaciones anidadas: «enfermedad de Dupuytren» aparece dentro de un fragmento más amplio etiquetado como procedimiento. Esto evidencia la necesidad de resolver los solapamientos antes de convertir las anotaciones al esquema BIO plano utilizado por el modelo.

El esquema de clasificación incluye **11 etiquetas BIO**: `O`, para tokens fuera de una entidad, y dos etiquetas por cada una de las cinco categorías: `B-`, que indica el inicio, e `I-`, que indica la continuación. La tabla muestra el catálogo completo de etiquetas disponibles; sus frecuencias por token se obtienen después de la tokenización y alineación del paso 5. Por tanto, los conteos presentados corresponden a anotaciones de entidades originales, no a cantidades de tokens BIO.

### PASO 04 — Partición fija por documento

Conservamos el algoritmo, la semilla 42 y la huella SHA-256 del notebook original: 600 documentos de entrenamiento y 150 de validación. Los dos modelos comparten estos documentos. Los 250 documentos de test no participan en selección de modelos ni hiperparámetros.


In [13]:
# PASO 04 - Particion unica compartida por los dos modelos y sus variantes (sin test)
rng = np.random.default_rng(CFG['split_seed'])
order = rng.permutation(official_train_ids).tolist()
n_sample = min(len(order), max(5, math.ceil(len(order) * CFG['fraction'])))
selected = order[:n_sample]
n_val = max(1, round(n_sample * CFG['validation_fraction']))
val_ids, train_ids = sorted(selected[:n_val]), sorted(selected[n_val:])
assert train_ids and val_ids
assert not set(train_ids) & set(val_ids)
assert not {fingerprint(texts[k]) for k in train_ids} & {fingerprint(texts[k]) for k in val_ids}
split_payload = {'train': train_ids, 'validation': val_ids, 'official_test': official_test_ids}
split_sha256 = hashlib.sha256(json.dumps(split_payload, sort_keys=True).encode()).hexdigest()
REPORT['split'] = {'sample_documents': n_sample, 'train_documents': len(train_ids),
                   'validation_documents': len(val_ids), 'test_used_for_training': False,
                   'split_seed': CFG['split_seed'], 'sha256': split_sha256}
(RUN / 'split.json').write_text(json.dumps({**split_payload, 'split_seed': CFG['split_seed'], 'sha256': split_sha256}, indent=2), encoding='utf-8')
display(Markdown('### PASO 04 — Partición por documento'))
mostrar_tabla_configuracion('Resumen de la partición', {
    'Documentos de la muestra (entrenamiento + validación)': n_sample,
    'Documentos de entrenamiento': len(train_ids),
    'Documentos de validación': len(val_ids),
    'Documentos de prueba reservados': len(official_test_ids),
    'Prueba utilizada para entrenar': REPORT['split']['test_used_for_training'],
    'Semilla de partición': CFG['split_seed'],
})

conteos_muestra = (df_train[df_train.doc_id.isin(selected)].label.value_counts()
                   .reindex(LABELS, fill_value=0))
lineas_muestra = [
    '#### Anotaciones por categoría en la muestra', '',
    '| Categoría | Anotaciones |', '| --- | ---: |',
]
lineas_muestra.extend(f'| {categoria} | {int(cantidad):,} |'
                      for categoria, cantidad in conteos_muestra.items())
lineas_muestra.append(f'| **Total** | **{int(conteos_muestra.sum()):,}** |')
display(Markdown('\n'.join(lineas_muestra)))

TALLER1_SPLIT_SHA256 = 'ab06b4e8dd54e120c2fed93f276f1944a6e67c5227281778c38add2eea4dc497'
mostrar_tabla_configuracion('Verificación de la huella SHA-256', {
    'Partición actual': split_sha256,
    'Partición del taller 1': TALLER1_SPLIT_SHA256,
    'Las huellas coinciden': split_sha256 == TALLER1_SPLIT_SHA256,
})
if CFG['fraction'] == 1.0 and CFG['validation_fraction'] == 0.20:
    assert split_sha256 == TALLER1_SPLIT_SHA256, 'La partición difiere de la referencia.'
if split_sha256 == TALLER1_SPLIT_SHA256:
    display(Markdown('**Verificación correcta:** la partición es idéntica a la del taller 1; '
                     'la comparación entre talleres utiliza los mismos documentos.'))
else:
    display(Markdown('**Aviso:** la huella de partición no coincide con la del taller 1. '
                     'Revisa `split_seed`, `fraction` y `validation_fraction` si esperabas '
                     'la misma partición. La comparación entre talleres dejaría de ser '
                     'sobre el mismo conjunto de documentos.'))

display(Markdown(
    f"**PASO 04 OK.** Los {len(CFG['checkpoints'])} modelos × "
    f"{len(CFG['variants'])} variantes × {len(CFG['training_seeds'])} semillas "
    'usarán estos mismos documentos. El entrenamiento todavía no ha comenzado.'
))


### PASO 04 — Partición por documento

#### Resumen de la partición

| Parámetro | Valor |
| --- | --- |
| Documentos de la muestra (entrenamiento + validación) | 750 |
| Documentos de entrenamiento | 600 |
| Documentos de validación | 150 |
| Documentos de prueba reservados | 250 |
| Prueba utilizada para entrenar | No |
| Semilla de partición | 42 |

#### Anotaciones por categoría en la muestra

| Categoría | Anotaciones |
| --- | ---: |
| CHEMICAL | 3,283 |
| DISEASE | 8,065 |
| PROCEDURE | 11,065 |
| PROTEIN | 2,253 |
| SYMPTOM | 9,091 |
| **Total** | **33,757** |

#### Verificación de la huella SHA-256

| Parámetro | Valor |
| --- | --- |
| Partición actual | ab06b4e8dd54e120c2fed93f276f1944a6e67c5227281778c38add2eea4dc497 |
| Partición del taller 1 | ab06b4e8dd54e120c2fed93f276f1944a6e67c5227281778c38add2eea4dc497 |
| Las huellas coinciden | Sí |

**Verificación correcta:** la partición es idéntica a la del taller 1; la comparación entre talleres utiliza los mismos documentos.

**PASO 04 OK.** Los 2 modelos × 2 variantes × 1 semillas usarán estos mismos documentos. El entrenamiento todavía no ha comenzado.

### PASO 05 — Referencia común y ventanas con contexto

Primero resolvemos las anotaciones sobre palabras de spaCy, con alineación estricta y política de entidad más larga. Este conjunto de referencia se calcula una sola vez y es idéntico para ambos modelos. Se conservan los offsets de caracteres para evaluar por documento.

Cada tokenizador procesa esas palabras con `is_split_into_words=True`; RoBERTa usa `add_prefix_space=True`, requerido para entradas pretokenizadas. Solo se supervisa el primer subtoken. Si algún tokenizador omite una palabra, la preparación falla en vez de reducir silenciosamente el conjunto evaluado.

Las ventanas tienen hasta 512 posiciones, límites de palabras y solapamiento objetivo de 64 subtokens. Cada palabra se asigna a una sola ventana, eligiendo aquella donde dispone de más contexto a ambos lados. El resto de sus apariciones aporta contexto, pero no pérdida ni predicciones duplicadas. La selección no consulta las etiquetas gold. Las entidades que atraviesan ventanas se reconstruyen por documento durante la evaluación.

Se reconstruyen las características para evitar reutilizar la caché anterior. Se guardan auditorías y una huella del gold común. El test solo se prepara si se activa expresamente el paso 11.


In [14]:
# PASO 05 - Gold común por palabras y ventanas con solapamiento real
import spacy
import transformers
from transformers import AutoTokenizer
from tqdm.auto import tqdm

t0 = time.perf_counter()
word_nlp = spacy.load('es_core_news_sm')
word_nlp.disable_pipes(*word_nlp.pipe_names)
REPORT['versions'].update(transformers=transformers.__version__, spacy=spacy.__version__,
                          spacy_model=word_nlp.meta.get('version'))
PREPROCESSING_POLICY = 'shared-spacy-first-subtoken-overlap-owner-document-v1'
tokenizers = {
    key: AutoTokenizer.from_pretrained(checkpoint, use_fast=True,
                                       **({'add_prefix_space': True} if key == 'clinical' else {}))
    for key, checkpoint in CFG['checkpoints'].items()
}
for key, tok in tokenizers.items():
    assert tok.is_fast and tok.num_special_tokens_to_add(pair=False) == 2
    assert CFG['max_length'] <= tok.model_max_length
    tok.padding_side = 'right'
    mostrar_tabla_configuracion(f'Tokenizador: {key}', {
        'Checkpoint': CFG['checkpoints'][key], 'Vocabulario': tok.vocab_size,
        'Primer subtoken supervisado': True, 'Longitud máxima': CFG['max_length'],
        'Solapamiento objetivo': CFG['overlap_subtokens'],
    })

def prepare_common_documents(ids, annotations):
    """Gold independiente del modelo; offsets estrictos y solapes flat-longest."""
    groups = {k: g for k, g in annotations.groupby('doc_id')}
    documents = {}
    stats = Counter(documents=len(ids), annotations=0, duplicates=0, unaligned=0,
                    overlap_excluded=0, retained=0, words=0, ignored_words=0)
    for doc_id, doc in zip(ids, word_nlp.pipe((texts[k] for k in ids), batch_size=16)):
        # Los espacios aislados no tienen representación léxica en BETO.
        tokens = [t for t in doc if not t.is_space]
        offsets = [(t.idx, t.idx + len(t)) for t in tokens]
        starts = {a: i for i, (a, b) in enumerate(offsets)}
        ends = {b: i + 1 for i, (a, b) in enumerate(offsets)}
        gold, candidates, ignored, seen = [0] * len(tokens), [], [], set()
        for row in groups.get(doc_id, annotations.iloc[:0]).itertuples(index=False):
            a, b, label = int(row.start_span), int(row.end_span), row.label
            stats['annotations'] += 1
            identity = (a, b, label)
            if identity in seen:
                stats['duplicates'] += 1
                continue
            seen.add(identity)
            if a not in starts or b not in ends or ends[b] <= starts[a]:
                stats['unaligned'] += 1
                ignored.append((a, b))
                continue
            candidates.append((a, b, label, starts[a], ends[b]))
        retained = []
        for a, b, label, start, end in sorted(candidates, key=lambda x: (-(x[1]-x[0]), x[0], x[2])):
            if any(gold[i] != 0 for i in range(start, end)):
                stats['overlap_excluded'] += 1
                ignored.append((a, b))
                continue
            gold[start] = tag_to_ix[f'B-{label}']
            gold[start+1:end] = [tag_to_ix[f'I-{label}']] * (end-start-1)
            retained.append([a, b, label])
            stats['retained'] += 1
        for i, (a, b) in enumerate(offsets):
            if gold[i] == 0 and any(a < y and b > x for x, y in ignored):
                gold[i] = IGNORE
        assert tokens, f'Documento vacío: {doc_id}'
        documents[doc_id] = {'doc_id': doc_id, 'words': [t.text for t in tokens],
                             'offsets': offsets, 'labels': gold, 'entities': sorted(retained)}
        stats['words'] += len(tokens)
        stats['ignored_words'] += gold.count(IGNORE)
    assert stats['annotations'] == sum(stats[k] for k in ('duplicates', 'unaligned', 'overlap_excluded', 'retained'))
    return documents, dict(stats)

def common_gold_hash(documents):
    payload = {k: {'offsets': v['offsets'], 'labels': v['labels'], 'entities': v['entities']}
               for k, v in documents.items()}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def build_word_windows(document, tokenizer):
    """Ventanas sin cortar palabras, con una sola posición propietaria por palabra."""
    encoding = tokenizer(document['words'], is_split_into_words=True,
                         add_special_tokens=False, truncation=False, verbose=False)
    ids, word_ids = encoding['input_ids'], encoding.word_ids()
    groups = {}
    for pos, wid in enumerate(word_ids):
        assert wid is not None
        groups.setdefault(wid, []).append(pos)
    assert sorted(groups) == list(range(len(document['words']))), (
        f"El tokenizador omitió palabras de {document['doc_id']}; no se permite cambiar el gold.")
    assert word_ids == sorted(word_ids), 'Orden de palabras inesperado.'
    capacity = CFG['max_length'] - 2
    windows, cursor = [], 0
    while cursor < len(groups):
        end, size = cursor, 0
        while end < len(groups) and size + len(groups[end]) <= capacity:
            size += len(groups[end]); end += 1
        if end == cursor:
            raise ValueError(f"Palabra mayor que la capacidad de ventana: {document['doc_id']}/{cursor}")
        left, right = groups[cursor][0], groups[end-1][-1]+1
        windows.append({'doc_id': document['doc_id'],
                        'input_ids': [tokenizer.cls_token_id] + ids[left:right] + [tokenizer.sep_token_id],
                        'attention_mask': [1] * (right-left+2),
                        'labels': [IGNORE] * (right-left+2),
                        'prediction_word_ids': [-1] * (right-left+2),
                        'start_word': cursor, 'end_word': end,
                        '_left': left, '_right': right})
        if end == len(groups):
            break
        target = right - CFG['overlap_subtokens']
        # Buscar DENTRO de la ventana actual: el original empezaba en end y no solapaba.
        cursor = next((w for w in range(cursor+1, end) if groups[w][0] >= target), end)
    owners = {}
    for wi, window in enumerate(windows):
        left, right = window['_left'], window['_right']
        for wid in range(window['start_word'], window['end_word']):
            context = min(groups[wid][0] - left, right - 1 - groups[wid][-1])
            if wid not in owners or context > owners[wid][0]:
                owners[wid] = (context, wi, 1 + groups[wid][0] - left)
    for wid, (_, wi, pos) in owners.items():
        windows[wi]['labels'][pos] = document['labels'][wid]
        windows[wi]['prediction_word_ids'][pos] = wid
    for window in windows:
        window.pop('_left'); window.pop('_right')
        assert len(window['input_ids']) <= CFG['max_length']
    # Evitar lotes cuya pérdida no tiene ninguna posición evaluable.
    windows = [w for w in windows if any(y != IGNORE for y in w['labels'])]
    supervised = [wid for w in windows for wid, y in zip(w['prediction_word_ids'], w['labels']) if y != IGNORE]
    expected = [i for i, y in enumerate(document['labels']) if y != IGNORE]
    assert sorted(supervised) == expected, 'Faltan palabras o hay supervisión duplicada.'
    return windows

def make_examples(checkpoint_key, documents, split_name):
    start = time.perf_counter()
    examples = []
    for document in tqdm(documents.values(), desc=f'Ventanas {checkpoint_key} {split_name}'):
        examples.extend(build_word_windows(document, tokenizers[checkpoint_key]))
    stats = {'documents': len(documents), 'sequences': len(examples),
             'supervised_words': sum(y != IGNORE for e in examples for y in e['labels']),
             'retained': sum(len(d['entities']) for d in documents.values()),
             'gold_sha256': common_gold_hash(documents), 'policy': PREPROCESSING_POLICY,
             'seconds_this_run': time.perf_counter() - start}
    (RUN / f'alignment_{checkpoint_key}_{split_name}.json').write_text(json.dumps(stats, indent=2), encoding='utf-8')
    return examples, stats

common_documents, REPORT['common_gold'] = {}, {}
for split_name, ids in [('train', train_ids), ('validation', val_ids)]:
    common_documents[split_name], audit = prepare_common_documents(ids, df_train)
    audit['sha256'] = common_gold_hash(common_documents[split_name])
    REPORT['common_gold'][split_name] = audit
    (RUN / f'common_gold_{split_name}.json').write_text(
        json.dumps(common_documents[split_name], ensure_ascii=False), encoding='utf-8')
    mostrar_tabla_configuracion(f'Referencia común — {split_name}', audit)
train_examples, val_examples, REPORT['alignment'] = {}, {}, {}
for checkpoint_key in CFG['checkpoints']:
    train_examples[checkpoint_key], train_stats = make_examples(checkpoint_key, common_documents['train'], 'train')
    val_examples[checkpoint_key], val_stats = make_examples(checkpoint_key, common_documents['validation'], 'validation')
    REPORT['alignment'][checkpoint_key] = {'train': train_stats, 'validation': val_stats}
    for split_name, stats in [('train', train_stats), ('validation', val_stats)]:
        assert stats['gold_sha256'] == REPORT['common_gold'][split_name]['sha256']
        mostrar_tabla_configuracion(f'{checkpoint_key} — {split_name}', stats)
REPORT['preprocessing_seconds'] = time.perf_counter() - t0
REPORT['preprocessing_policy'] = PREPROCESSING_POLICY
mostrar_tabla('PASO 05 OK — Ventanas por modelo', ['Modelo', 'Entrenamiento', 'Validación'],
              [[k, len(train_examples[k]), len(val_examples[k])] for k in CFG['checkpoints']])


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

...

Ventanas clinical train:   0%|          | 0/600 [00:00<?, ?it/s]

Ventanas clinical validation:   0%|          | 0/150 [00:00<?, ?it/s]

#### clinical — train

| Parámetro | Valor |
| --- | --- |
| documents | 600 |
| sequences | 836 |
| supervised_words | 238187 |
| retained | 22401 |
| gold_sha256 | e93757b22c46fb8079377181928694a20441d5691d7ebe1d046a9e34971e5c34 |
| policy | shared-spacy-first-subtoken-overlap-owner-document-v1 |
| seconds_this_run | 1.8472300529999757 |

#### clinical — validation

| Parámetro | Valor |
| --- | --- |
| documents | 150 |
| sequences | 219 |
| supervised_words | 61938 |
| retained | 5861 |
| gold_sha256 | f370133e86e6c1dbd2a537e9295a9506fc992f4a254b387f08a34655860790ab |
| policy | shared-spacy-first-subtoken-overlap-owner-document-v1 |
| seconds_this_run | 0.4992725930001143 |

#### PASO 05 OK — Ventanas por modelo

| Modelo | Entrenamiento | Validación |
| --- | --- | --- |
| beto | 947 | 250 |
| clinical | 836 | 219 |

### PASO 06 — Datasets y pérdida sin ponderación

Los datasets contienen únicamente entradas del modelo y etiquetas. La correspondencia entre ventanas, palabras y documentos se conserva aparte para las métricas. Usamos la entropía cruzada del modelo, sin pesos por clase; cada palabra evaluable aporta una sola posición supervisada. Las continuaciones, repeticiones de contexto y padding usan `IGNORE=-100`. Para esta comparación inicial se mantiene `class_weights=False`.


In [15]:
# PASO 06 - Datasets sin metadatos y pérdida estándar del modelo
from datasets import Dataset
from transformers import DataCollatorForTokenClassification

assert CFG['class_weights'] is False, 'Esta prueba usa pérdida sin ponderación; mantén class_weights=False.'
MODEL_COLUMNS = ('input_ids', 'attention_mask', 'labels')
def model_features(examples):
    return [{k: e[k] for k in MODEL_COLUMNS} for e in examples]

hf_datasets, collators = {}, {}
REPORT['vocabulary'] = {}
for checkpoint_key in CFG['checkpoints']:
    hf_datasets[checkpoint_key] = {
        'train': Dataset.from_list(model_features(train_examples[checkpoint_key])),
        'validation': Dataset.from_list(model_features(val_examples[checkpoint_key])),
    }
    collators[checkpoint_key] = DataCollatorForTokenClassification(
        tokenizer=tokenizers[checkpoint_key], label_pad_token_id=IGNORE)
    counts = Counter(y for e in train_examples[checkpoint_key] for y in e['labels'] if y != IGNORE)
    common_counts = Counter(y for d in common_documents['train'].values() for y in d['labels'] if y != IGNORE)
    assert counts == common_counts, 'La supervisión por palabra debe ser idéntica entre modelos.'
    REPORT['vocabulary'][checkpoint_key] = {
        'train_tag_counts': {tag: counts[i] for i, tag in enumerate(TAGS)},
        'missing_train_tags': [tag for i, tag in enumerate(TAGS) if not counts[i]],
        'tokenizer_vocab_size': tokenizers[checkpoint_key].vocab_size,
        'train_blocks': len(train_examples[checkpoint_key]),
        'validation_blocks': len(val_examples[checkpoint_key]), 'class_weights': False,
    }
    mostrar_tabla_configuracion(f'{checkpoint_key} — Datasets', {
        'Ventanas de entrenamiento': len(train_examples[checkpoint_key]),
        'Ventanas de validación': len(val_examples[checkpoint_key]),
        'Pérdida': 'Entropía cruzada sin ponderación', 'Padding de etiquetas': IGNORE,
    })
    mostrar_tabla(f'{checkpoint_key} — Etiquetas BIO', ['ID', 'Etiqueta', 'Palabras supervisadas', 'Peso'],
                  [[i, tag, counts[i], 1.0] for i, tag in enumerate(TAGS)])
    if REPORT['vocabulary'][checkpoint_key]['missing_train_tags']:
        mostrar_tabla_configuracion('Etiquetas ausentes', {'Etiquetas': REPORT['vocabulary'][checkpoint_key]['missing_train_tags']})
display(Markdown('### PASO 06 OK\n\nAmbos modelos supervisan exactamente las mismas palabras y etiquetas.'))


#### beto — Datasets

| Parámetro | Valor |
| --- | --- |
| Ventanas de entrenamiento | 947 |
| Ventanas de validación | 250 |
| Pérdida | Entropía cruzada sin ponderación |
| Padding de etiquetas | -100 |

#### beto — Etiquetas BIO

| ID | Etiqueta | Palabras supervisadas | Peso |
| --- | --- | --- | --- |
| 0 | O | 164593 | 1.0 |
| 1 | B-CHEMICAL | 1867 | 1.0 |
| 2 | I-CHEMICAL | 251 | 1.0 |
| 3 | B-DISEASE | 5866 | 1.0 |
| 4 | I-DISEASE | 13097 | 1.0 |
| 5 | B-PROCEDURE | 7253 | 1.0 |
| 6 | I-PROCEDURE | 16851 | 1.0 |
| 7 | B-PROTEIN | 1090 | 1.0 |
| 8 | I-PROTEIN | 336 | 1.0 |
| 9 | B-SYMPTOM | 6325 | 1.0 |
| 10 | I-SYMPTOM | 20658 | 1.0 |

#### clinical — Datasets

| Parámetro | Valor |
| --- | --- |
| Ventanas de entrenamiento | 836 |
| Ventanas de validación | 219 |
| Pérdida | Entropía cruzada sin ponderación |
| Padding de etiquetas | -100 |

#### clinical — Etiquetas BIO

| ID | Etiqueta | Palabras supervisadas | Peso |
| --- | --- | --- | --- |
| 0 | O | 164593 | 1.0 |
| 1 | B-CHEMICAL | 1867 | 1.0 |
| 2 | I-CHEMICAL | 251 | 1.0 |
| 3 | B-DISEASE | 5866 | 1.0 |
| 4 | I-DISEASE | 13097 | 1.0 |
| 5 | B-PROCEDURE | 7253 | 1.0 |
| 6 | I-PROCEDURE | 16851 | 1.0 |
| 7 | B-PROTEIN | 1090 | 1.0 |
| 8 | I-PROTEIN | 336 | 1.0 |
| 9 | B-SYMPTOM | 6325 | 1.0 |
| 10 | I-SYMPTOM | 20658 | 1.0 |

### PASO 06 OK

Ambos modelos supervisan exactamente las mismas palabras y etiquetas.

### PASO 07 — Modelos y evaluación común por documento

Reconstruimos una etiqueta por palabra en cada documento, usando únicamente la ventana propietaria de esa palabra. Después convertimos las entidades BIO a offsets de caracteres y exigimos coincidencia exacta de documento, inicio, fin y categoría. Las continuaciones de subtokens no introducen etiquetas `O` artificiales dentro de entidades. Los huecos del gold realmente excluidos sí separan entidades.

Comprobamos que las entidades reconstruidas del gold coincidan exactamente con las anotaciones retenidas, independientemente del modelo. Se calcula F1 micro y métricas por categoría; la exactitud por palabra es secundaria. Una `I-` sin inicio compatible se trata como inicio de entidad y se registra como transición inválida.


In [16]:
# PASO 07 - Modelos, variante congelada/fine-tuned y metricas de NER
from transformers import (AutoModelForTokenClassification, BertForTokenClassification,
                          RobertaForTokenClassification, Trainer)
from safetensors.torch import load_file as load_safetensors

assert transformers.__version__ == '5.6.0', 'Ejecuta el paso 01 y reinicia la sesión antes de continuar.'

class CanonicalCheckpointMixin:
    def save_pretrained(self, *args, **kwargs):
        # Transformers 5 puede revertir los nombres legacy al guardar; Trainer
        # restaura con load_state_dict y necesita los nombres de la arquitectura.
        kwargs['save_original_format'] = False
        return super().save_pretrained(*args, **kwargs)

class CanonicalBertForTokenClassification(CanonicalCheckpointMixin, BertForTokenClassification):
    pass

class CanonicalRobertaForTokenClassification(CanonicalCheckpointMixin, RobertaForTokenClassification):
    pass

MODEL_CLASSES = {'beto': CanonicalBertForTokenClassification,
                 'clinical': CanonicalRobertaForTokenClassification}

def validate_loading_info(model, info, allow_new_head=False):
    missing = sorted(info.get('missing_keys', []))
    unexpected = sorted(info.get('unexpected_keys', []))
    allowed_missing = {k for k in model.state_dict() if k.startswith('classifier.')} if allow_new_head else set()
    bad_missing = set(missing) - allowed_missing
    # Las cabezas de preentrenamiento no se reutilizan en NER; position_ids era
    # un buffer persistente en checkpoints antiguos, no un parámetro aprendido.
    allowed_unexpected = lambda k: allow_new_head and (
        k.startswith(('cls.', 'lm_head.', 'bert.pooler.', 'roberta.pooler.'))
        or k in ('bert.embeddings.position_ids', 'roberta.embeddings.position_ids'))
    bad_unexpected = [k for k in unexpected if not allowed_unexpected(k)]
    if bad_missing or bad_unexpected or info.get('mismatched_keys') or info.get('error_msgs'):
        raise RuntimeError(f'Carga incompleta: faltantes={sorted(bad_missing)}, inesperadas={bad_unexpected}, '
                           f'dimensiones={info.get("mismatched_keys")}, errores={info.get("error_msgs")}')
    for name, parameter in model.named_parameters():
        if not torch.isfinite(parameter).all():
            raise RuntimeError(f'Parámetro no finito al cargar: {name}')
    return {'encoder_completo': True, 'cabezal_nuevo': missing,
            'pesos_de_preentrenamiento_no_usados': unexpected}

def load_model(checkpoint_key):
    model, info = MODEL_CLASSES[checkpoint_key].from_pretrained(
        CFG['checkpoints'][checkpoint_key], num_labels=len(TAGS), id2label=id2label,
        label2id=label2id, dtype=torch.float32, output_loading_info=True)
    audit = validate_loading_info(model, info, allow_new_head=True)
    REPORT.setdefault('model_loading', {})[checkpoint_key] = audit
    return model

def restore_checkpoint_strict(model, checkpoint_dir):
    """Restauración completa, sin tolerar parámetros faltantes o nombres legacy."""
    directory = Path(checkpoint_dir)
    index_path = directory / 'model.safetensors.index.json'
    if index_path.exists():
        names = sorted(set(json.loads(index_path.read_text())['weight_map'].values()))
    else:
        names = ['model.safetensors']
    state = {}
    for name in names:
        shard = load_safetensors(str(directory / name), device='cpu')
        if state.keys() & shard.keys():
            raise RuntimeError('Claves repetidas entre shards del checkpoint.')
        state.update(shard)
    expected = model.state_dict()
    missing, unexpected = set(expected) - set(state), set(state) - set(expected)
    if missing or unexpected:
        raise RuntimeError(f'Checkpoint incompatible: faltantes={sorted(missing)}, '
                           f'inesperadas={sorted(unexpected)}. No reutilices checkpoints de la corrida anterior.')
    for name, value in state.items():
        if value.shape != expected[name].shape or not torch.isfinite(value).all():
            raise RuntimeError(f'Peso inválido en el checkpoint: {name}')
    model.load_state_dict(state, strict=True)
    return {'restauracion_estricta': True, 'tensores_restaurados': len(state)}

class VerifiedCheckpointTrainer(Trainer):
    def _load_best_model(self):
        # Mismo camino probado en el PASO 08. Este notebook usa un proceso
        # con DataParallel; no configura FSDP, DeepSpeed ni adaptadores PEFT.
        self.checkpoint_restore_audit = restore_checkpoint_strict(self.model, self.state.best_model_checkpoint)

def load_saved_model(checkpoint_dir):
    model, info = AutoModelForTokenClassification.from_pretrained(
        checkpoint_dir, local_files_only=True, dtype=torch.float32, output_loading_info=True)
    validate_loading_info(model, info, allow_new_head=False)
    return model

def apply_variant(model, variant):
    """Aplica la variante `frozen` (congela el encoder base; solo se entrena el
    cabezal de clasificacion) o `fine_tuned` (todos los pesos entrenables)."""
    if variant == 'frozen':
        for param in model.base_model.parameters():
            param.requires_grad = False
    elif variant != 'fine_tuned':
        raise ValueError(f'Variante desconocida: {variant}')
    return model

def entity_spans(tag_ids):
    """Reconstruye spans BIO en una secuencia de palabras.

    Una I- sin entidad compatible inicia una entidad y cuenta como transición
    inválida. Los spans se convierten después a offsets del documento; compartir
    esta función no basta para comparar protocolos con distintos gold o ventanas.
    """
    result, active, begin, invalid = set(), None, None, 0
    for i, idx in enumerate(list(tag_ids) + [0]):
        tag = TAGS[idx] if idx >= 0 else 'O'
        prefix, label = tag.split('-', 1) if tag != 'O' else ('O', None)
        continuing = prefix == 'I' and label == active
        if active is not None and not continuing:
            result.add((begin, i, active)); active = None
        if prefix in ('B', 'I') and not continuing:
            invalid += int(prefix == 'I')
            begin, active = i, label
    return result, invalid

def scores_from_counts(tp, predicted, gold):
    """Calcula precision, recall y F1 a partir de conteos agregados: verdaderos
    positivos (tp), total de entidades predichas y total de entidades reales (gold)
    -- identica a la de taller 1."""
    precision = tp / predicted if predicted else 0.0
    recall = tp / gold if gold else 0.0
    return {'precision': precision, 'recall': recall,
            'f1': 2*precision*recall/(precision+recall) if precision+recall else 0.0,
            'support': gold, 'predicted': predicted}


def reconstruct_word_predictions(predictions, examples, documents):
    """Una predicción por palabra; el orden corresponde al dataset de evaluación."""
    assert len(predictions) == len(examples), 'Orden/tamaño de evaluación incompatible.'
    result = {k: [None] * len(d['words']) for k, d in documents.items()}
    for row, example in zip(predictions, examples):
        assert len(row) >= len(example['input_ids'])
        target = result[example['doc_id']]
        for pos, wid in enumerate(example['prediction_word_ids']):
            if wid < 0:
                continue
            assert target[wid] is None, 'Predicción duplicada para una palabra.'
            target[wid] = int(row[pos])
    for doc_id, doc in documents.items():
        for wid, label in enumerate(doc['labels']):
            if label == IGNORE:
                result[doc_id][wid] = 0
            else:
                assert result[doc_id][wid] is not None, f'Palabra sin predicción: {doc_id}/{wid}'
    return result

def character_entities(tag_ids, offsets):
    spans, invalid = entity_spans(tag_ids)
    return {(offsets[a][0], offsets[b-1][1], label) for a, b, label in spans}, invalid

def make_document_metrics(examples, documents):
    def compute_metrics(eval_pred):
        logits = eval_pred.predictions
        if isinstance(logits, tuple):
            logits = logits[0]
        predictions = np.argmax(logits, axis=-1) if logits.ndim == 3 else logits
        word_predictions = reconstruct_word_predictions(predictions, examples, documents)
        totals = {label: Counter() for label in LABELS}
        invalid = correct = tokens = 0
        for doc_id, doc in documents.items():
            predicted = word_predictions[doc_id]
            gold = [y if y != IGNORE else 0 for y in doc['labels']]
            ps, errors = character_entities(predicted, doc['offsets'])
            gs, gold_errors = character_entities(gold, doc['offsets'])
            assert gold_errors == 0 and gs == {tuple(e) for e in doc['entities']}, 'Gold reconstruido distinto.'
            invalid += errors
            for pred, ref in zip(predicted, doc['labels']):
                if ref != IGNORE:
                    correct += int(pred == ref); tokens += 1
            for label in LABELS:
                totals[label].update(tp=sum(e[2] == label for e in ps & gs),
                                     predicted=sum(e[2] == label for e in ps),
                                     gold=sum(e[2] == label for e in gs))
        micro = scores_from_counts(sum(c['tp'] for c in totals.values()),
                                   sum(c['predicted'] for c in totals.values()),
                                   sum(c['gold'] for c in totals.values()))
        result = {k: v for k, v in micro.items()}
        result.update(accuracy=correct/tokens if tokens else 0.0, invalid_bio_transitions=invalid)
        for label, counts in totals.items():
            metrics = scores_from_counts(counts['tp'], counts['predicted'], counts['gold'])
            for name, value in metrics.items():
                result[f'{name}_{label}'] = value
        return result
    return compute_metrics

def preprocess_logits_for_metrics(logits, labels):
    # Reducir memoria de evaluación: no conservar todos los logits de las ventanas.
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

validation_metrics_fns = {k: make_document_metrics(val_examples[k], common_documents['validation'])
                          for k in CFG['checkpoints']}
REPORT['metric_scope'] = 'F1 micro de entidades exactas por documento y offsets de caracteres, gold común.'
display(Markdown('### PASO 07 OK\n\nModelos y métricas por documento preparados.'))


### PASO 07 OK

Modelos y métricas por documento preparados.

### PASO 08 — Comprobación técnica de los cuatro escenarios

Se usan exclusivamente ventanas de entrenamiento con entidades. Para cada modelo y variante se comprueba:

1. Carga completa del encoder y valores finitos; solo el cabezal NER puede inicializarse desde cero.
2. Pérdida/logits finitos y gradientes no nulos. En `frozen`, ningún peso del encoder cambia; en `fine_tuned`, debe cambiar al menos uno. El cabezal debe actualizarse en ambos casos.
3. Guardado con nombres actuales y restauración estricta mediante el mismo método usado para el mejor checkpoint. Después se carga además una instancia nueva: sus pesos y logits deben coincidir con los anteriores al guardado.
4. Descenso de pérdida y predicciones distintas de `O` después de 20 pasos. Estos dos indicadores se muestran por separado: una alerta no equivale a un error de serialización, pero tampoco permite afirmar que el modelo ya reconoce entidades correctamente.

Los errores de integridad bloquean el paso 09 y se guardan en `smoke_test.json`. Las alertas de aprendizaje quedan visibles y el paso 09 vuelve a comprobar si se predicen entidades en validación. Ninguna prueba de un lote garantiza un buen F1 en documentos nuevos. Los pesos temporales se descartan y el entrenamiento comienza nuevamente desde el checkpoint preentrenado.


In [17]:
# PASO 08 - Carga, gradientes, actualización y guardado/recarga para los cuatro escenarios
from transformers import set_seed
import gc
import tempfile

def parameter_fingerprint(parameters):
    digest = hashlib.sha256()
    for name, parameter in parameters:
        digest.update(name.encode())
        digest.update(parameter.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

def check_checkpoint_roundtrip(model, batch):
    """Verifica restauración del Trainer y carga independiente, sin entrenamiento."""
    inputs = {k: v for k, v in batch.items() if k != 'labels'}
    model.eval()
    with torch.no_grad():
        before = model(**inputs).logits.detach().cpu()
    reloaded = None
    try:
        with tempfile.TemporaryDirectory(prefix='checkpoint_check_', dir=RUN) as directory:
            model.save_pretrained(directory)
            # Perturbar LayerNorm obliga a comprobar que se restaura realmente;
            # no basta con recargar sobre parámetros que ya eran iguales.
            norm_parameter = next(p for name, p in model.named_parameters() if 'LayerNorm.weight' in name)
            with torch.no_grad():
                norm_parameter.add_(0.125)
            audit = restore_checkpoint_strict(model, directory)
            with torch.no_grad():
                restored = model(**inputs).logits.detach().cpu()
            torch.testing.assert_close(restored, before, rtol=1e-5, atol=1e-5)
            reloaded = load_saved_model(directory).to(next(model.parameters()).device)
            reloaded.eval()
            old_state, new_state = model.state_dict(), reloaded.state_dict()
            assert old_state.keys() == new_state.keys()
            assert all(torch.equal(old_state[k], new_state[k]) for k in old_state), 'Los pesos cambiaron al recargar.'
            with torch.no_grad():
                after = reloaded(**inputs).logits.detach().cpu()
            torch.testing.assert_close(after, before, rtol=1e-5, atol=1e-5)
            assert torch.equal(after.argmax(-1), before.argmax(-1)), 'Las etiquetas cambiaron al recargar.'
            return {**audit, 'pesos_identicos': True, 'predicciones_identicas': True,
                    'diferencia_maxima_logits': float((after-before).abs().max())}
    finally:
        del reloaded
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def run_smoke_test(checkpoint_key, variant):
    set_seed(CFG['training_seeds'][0])
    candidates = [e for e in train_examples[checkpoint_key] if any(y > 0 for y in e['labels'])]
    assert candidates, 'No hay entidades supervisadas en entrenamiento.'
    # Evitar un lote dominado por O para el diagnóstico; no altera el dataset real.
    candidates.sort(key=lambda e: sum(y > 0 for y in e['labels']) / sum(y != IGNORE for y in e['labels']), reverse=True)
    batch = collators[checkpoint_key](model_features(candidates[:2]))
    batch = {k: v.to(device) for k, v in batch.items()}
    model = apply_variant(load_model(checkpoint_key), variant).to(device)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                                 lr=CFG['learning_rate'][variant], weight_decay=CFG['weight_decay'])
    model.eval()  # Sin dropout; se permiten gradientes para la prueba del lote.
    encoder_before = parameter_fingerprint(model.base_model.named_parameters())
    head_before = parameter_fingerprint(model.classifier.named_parameters())
    losses = []
    try:
        for _ in range(CFG['smoke_steps']):
            optimizer.zero_grad(set_to_none=True)
            outputs = model(**batch)
            loss = outputs.loss
            assert torch.isfinite(loss) and torch.isfinite(outputs.logits).all(), 'Salida no finita.'
            loss.backward()
            norm = torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad), 1.0, error_if_nonfinite=True)
            assert norm > 0, 'Gradiente nulo.'
            if variant == 'frozen':
                assert all(not p.requires_grad and p.grad is None for p in model.base_model.parameters()), 'Encoder no congelado.'
            else:
                assert any(p.grad is not None and torch.count_nonzero(p.grad) > 0
                           for p in model.base_model.parameters()), 'Encoder sin gradiente en fine_tuned.'
            optimizer.step()
            losses.append(float(loss.item()))
        with torch.no_grad():
            final = model(**batch)
            assert torch.isfinite(final.loss) and torch.isfinite(final.logits).all()
            final_loss = float(final.loss.item())
            predictions = final.logits.argmax(-1)
        valid = batch['labels'] != IGNORE
        predicted_entity_words = int(((predictions > 0) & valid).sum().item())
        entity_words = int(((batch['labels'] > 0) & valid).sum().item())
        entity_correct = int(((predictions == batch['labels']) & (batch['labels'] > 0)).sum().item())
        encoder_changed = encoder_before != parameter_fingerprint(model.base_model.named_parameters())
        head_changed = head_before != parameter_fingerprint(model.classifier.named_parameters())
        assert head_changed, 'El cabezal no se actualizó.'
        assert encoder_changed == (variant == 'fine_tuned'), 'Actualización del encoder incompatible con la variante.'
        roundtrip = check_checkpoint_roundtrip(model, batch)
        alerts = []
        if final_loss >= losses[0]:
            alerts.append('La pérdida final del lote no disminuyó.')
        if not predicted_entity_words:
            alerts.append('El modelo predice solo O en las posiciones supervisadas del lote.')
        elif not entity_correct:
            alerts.append('Predice etiquetas de entidad, pero ninguna coincide con las etiquetas de entidad del lote.')
        return {'passed': True, 'status': 'OK_TECNICO_CON_ALERTAS' if alerts else 'OK_TECNICO',
                'steps': CFG['smoke_steps'], 'learning_rate': CFG['learning_rate'][variant],
                'initial_loss': losses[0], 'final_loss': final_loss, 'min_loss': min(losses+[final_loss]),
                'loss_decreased': final_loss < losses[0], 'encoder_updated': encoder_changed,
                'head_updated': head_changed, 'gold_entity_words': entity_words,
                'predicted_entity_words': predicted_entity_words, 'correct_entity_word_labels': entity_correct,
                'checkpoint': roundtrip, 'alerts': alerts}
    finally:
        del optimizer, model, batch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

REPORT.pop('smoke_test', None)
smoke_results = {}
for checkpoint_key in CFG['checkpoints']:
    for variant in CFG['variants']:
        name = f'{checkpoint_key}_{variant}'
        try:
            result = run_smoke_test(checkpoint_key, variant)
        except Exception as exc:
            smoke_results[name] = {'passed': False, 'status': 'ERROR_TECNICO',
                                   'error': str(exc), 'traceback': traceback.format_exc()}
            REPORT['smoke_test'] = {'passed': False, 'variants': smoke_results}
            (RUN / 'smoke_test.json').write_text(json.dumps(REPORT['smoke_test'], indent=2), encoding='utf-8')
            mostrar_tabla_configuracion(f'ERROR — {name}', smoke_results[name])
            raise
        smoke_results[name] = result
        mostrar_tabla_configuracion(name, result)
        for alert in result['alerts']:
            display(Markdown(f'**Alerta de aprendizaje ({name}):** {alert}'))
REPORT['smoke_test'] = {'passed': all(r['passed'] for r in smoke_results.values()), 'variants': smoke_results}
(RUN / 'smoke_test.json').write_text(json.dumps(REPORT['smoke_test'], indent=2), encoding='utf-8')
display(Markdown('### PASO 08 — Integridad técnica comprobada\n\nRevisa las alertas de aprendizaje anteriores. '
                 'Esta prueba no garantiza un buen F1 de validación. Los pesos temporales se descartan.'))


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CanonicalBertForTokenClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream ta

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

[transformers] CanonicalRobertaForTokenClassification LOAD REPORT from: PlanTL-GOB-ES/bsc-bio-ehr-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

#### clinical_fine_tuned

| Parámetro | Valor |
| --- | --- |
| passed | Sí |
| status | OK_TECNICO |
| steps | 20 |
| learning_rate | 2e-05 |
| initial_loss | 2.4696004390716553 |
| final_loss | 0.46429672837257385 |
| min_loss | 0.46429672837257385 |
| loss_decreased | Sí |
| encoder_updated | Sí |
| head_updated | Sí |
| gold_entity_words | 104 |
| predicted_entity_words | 102 |
| correct_entity_word_labels | 92 |
| checkpoint.restauracion_estricta | Sí |
| checkpoint.tensores_restaurados | 199 |
| checkpoint.pesos_identicos | Sí |
| checkpoint.predicciones_identicas | Sí |
| checkpoint.diferencia_maxima_logits | 0.0 |
| alerts |  |

### PASO 08 — Integridad técnica comprobada

Revisa las alertas de aprendizaje anteriores. Esta prueba no garantiza un buen F1 de validación. Los pesos temporales se descartan.

### PASO 09 — Cuatro entrenamientos comparables

BETO y RoBERTa se ejecutan con `frozen` (`1e-3`) y `fine_tuned` (`2e-5`), semilla 123. Ambos usan las mismas palabras y entidades, hasta 512 posiciones, lote 8 por GPU, 8 épocas máximas, paciencia 2 y warmup del 10 %. La tasa es la misma entre modelos dentro de cada estrategia.

Se guardan nombres actuales de parámetros y el mejor checkpoint se restaura de forma estricta. Se registra la auditoría de recuperación. Se avisa explícitamente si una evaluación produce cero entidades. Pérdida, precisión, recall y F1 quedan en Markdown y en los historiales CSV. Los cuatro entrenamientos se realizan en Kaggle; no se usa test.


In [18]:
# PASO 09 - Cuatro escenarios: dos modelos por dos estrategias
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, TrainerCallback, set_seed

assert REPORT.get('smoke_test', {}).get('passed'), 'Ejecuta primero el paso 08.'
for key in CFG['checkpoints']:
    for variant in CFG['variants']:
        probe = REPORT['smoke_test']['variants'].get(f'{key}_{variant}', {})
        assert probe.get('passed') and probe.get('learning_rate') == CFG['learning_rate'][variant], (
            'La prueba técnica no corresponde a esta configuración; repite el paso 08.')

class CheckFiniteTraining(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and metrics.get('eval_predicted') == 0:
            display(Markdown(f"**Alerta en época {state.epoch}:** cero entidades predichas en validación. "
                             'El F1 es cero; revisa esta corrida antes de interpretar la comparación.'))

    def on_log(self, args, state, control, logs=None, **kwargs):
        # FP16 puede omitir pasos por desbordamiento de gradientes; se registra grad_norm.
        for name in ('loss', 'eval_loss'):
            if logs and name in logs and not math.isfinite(float(logs[name])):
                raise RuntimeError(f'{name} no finito. La corrida se detiene; revisa learning_rate.')

def extract_metrics(metrics):
    return {'micro': {k: float(metrics[f'eval_{k}']) for k in ('precision', 'recall', 'f1')}
            | {'support': int(metrics['eval_support']), 'predicted': int(metrics['eval_predicted'])},
            'per_label': {label: {k: float(metrics[f'eval_{k}_{label}'])
                                 for k in ('precision', 'recall', 'f1')}
                          | {'support': int(metrics[f'eval_support_{label}'])} for label in LABELS},
            'loss': float(metrics['eval_loss']), 'accuracy': float(metrics['eval_accuracy']),
            'invalid_bio_transitions': int(metrics['eval_invalid_bio_transitions'])}

def train_run(checkpoint_key, variant, training_seed):
    run_dir = RUN / checkpoint_key / variant / f'seed_{training_seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    set_seed(training_seed)
    model = apply_variant(load_model(checkpoint_key), variant)
    parameters = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    args = TrainingArguments(
        output_dir=str(run_dir), seed=training_seed, data_seed=training_seed,
        num_train_epochs=CFG['epochs'][variant], learning_rate=CFG['learning_rate'][variant],
        per_device_train_batch_size=CFG['batch_size'], per_device_eval_batch_size=CFG['batch_size'],
        weight_decay=CFG['weight_decay'], warmup_steps=CFG['warmup_steps'],
        fp16=torch.cuda.is_available(), max_grad_norm=1.0,
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model='f1', greater_is_better=True,
        logging_strategy='steps', logging_steps=10, logging_first_step=True,
        logging_nan_inf_filter=False, report_to=[], disable_tqdm=False,
    )
    trainer = VerifiedCheckpointTrainer(model=model, args=args,
                      train_dataset=hf_datasets[checkpoint_key]['train'],
                      eval_dataset=hf_datasets[checkpoint_key]['validation'],
                      data_collator=collators[checkpoint_key], processing_class=tokenizers[checkpoint_key],
                      compute_metrics=validation_metrics_fns[checkpoint_key],
                      preprocess_logits_for_metrics=preprocess_logits_for_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=CFG['patience']), CheckFiniteTraining()])
    mostrar_tabla_configuracion(f'Entrenando {checkpoint_key} / {variant}', {
        'Semilla': training_seed, 'Tasa inicial': CFG['learning_rate'][variant],
        'Parámetros entrenables': trainable, 'Parámetros totales': parameters,
        'Épocas máximas': CFG['epochs'][variant], 'Lote por GPU': CFG['batch_size'],
        'Ventanas train': len(train_examples[checkpoint_key]), 'Ventanas validación': len(val_examples[checkpoint_key]),
    })
    try:
        train_output = trainer.train()
        metrics = trainer.evaluate()
        best_dir = run_dir / 'best_model'
        trainer.save_model(str(best_dir))
        best_step = int(Path(trainer.state.best_model_checkpoint).name.rsplit('-', 1)[-1])
        best_epoch = next((r.get('epoch') for r in trainer.state.log_history
                           if r.get('step') == best_step and 'eval_f1' in r), None)
        history = [{k: (str(v) if isinstance(v, float) and not math.isfinite(v) else v)
                    for k, v in row.items() if isinstance(v, (int, float, str, bool))}
                   for row in trainer.state.log_history]
        result = {'checkpoint': checkpoint_key, 'variant': variant, 'seed': training_seed,
                  'learning_rate': CFG['learning_rate'][variant], 'split_sha256': split_sha256,
                  'gold_sha256': REPORT['common_gold']['validation']['sha256'],
                  'parameters': parameters, 'trainable_parameters': trainable,
                  'checkpoint_dir': str(best_dir), 'best_epoch': best_epoch,
                  'best_metric_f1': float(trainer.state.best_metric),
                  'validation': extract_metrics(metrics),
                  'train_runtime_seconds': float(train_output.metrics['train_runtime']),
                  'epochs_completed': float(trainer.state.epoch), 'optimizer_steps': trainer.state.global_step,
                  'history': history, 'checkpoint_restore': trainer.checkpoint_restore_audit}
        (run_dir / 'result.json').write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
        pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
        eval_rows = [r for r in history if 'eval_f1' in r]
        mostrar_tabla(f'{checkpoint_key} — Evolución de validación',
                      ['Época', 'Pérdida', 'Precisión %', 'Recall %', 'F1 %'],
                      [[round(r['epoch'], 2), round(r['eval_loss'], 4), round(100*r['eval_precision'], 2),
                        round(100*r['eval_recall'], 2), round(100*r['eval_f1'], 2)] for r in eval_rows])
        return result
    finally:
        del trainer, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

REPORT['experiments'] = {}
for checkpoint_key in CFG['checkpoints']:
    REPORT['experiments'][checkpoint_key] = {}
    for variant in CFG['variants']:
        REPORT['experiments'][checkpoint_key][variant] = {}
        for training_seed in CFG['training_seeds']:
            result = train_run(checkpoint_key, variant, training_seed)
            REPORT['experiments'][checkpoint_key][variant][str(training_seed)] = result
            (RUN / 'experiments_progress.json').write_text(json.dumps(REPORT['experiments'], indent=2), encoding='utf-8')
n_runs = len(CFG['checkpoints']) * len(CFG['variants']) * len(CFG['training_seeds'])
display(Markdown(f'### PASO 09 OK\n\nFinalizaron **{n_runs} entrenamientos**. Continúa con el paso 10.'))


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CanonicalBertForTokenClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream ta

#### Entrenando beto / frozen

| Parámetro | Valor |
| --- | --- |
| Semilla | 123 |
| Tasa inicial | 0.001 |
| Parámetros entrenables | 8459 |
| Parámetros totales | 109268747 |
| Épocas máximas | 8 |
| Lote por GPU | 8 |
| Ventanas train | 947 |
| Ventanas validación | 250 |

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Support,Predicted,Accuracy,Invalid Bio Transitions,Precision Chemical,Recall Chemical,F1 Chemical,Support Chemical,Predicted Chemical,Precision Disease,Recall Disease,F1 Disease,Support Disease,Predicted Disease,Precision Procedure,Recall Procedure,F1 Procedure,Support Procedure,Predicted Procedure,Precision Protein,Recall Protein,F1 Protein,Support Protein,Predicted Protein,Precision Symptom,Recall Symptom,F1 Symptom,Support Symptom,Predicted Symptom
1,1.479303,1.386677,0.217939,0.256611,0.235700,5861,6901,0.763296,3507,0.762376,0.266898,0.395379,577,202,0.197704,0.334292,0.248464,1391,2352,0.296496,0.346275,0.319458,1906,2226,0.000000,0.000000,0.000000,305,0,0.106082,0.133769,0.118328,1682,2121
2,1.277214,1.150648,0.284844,0.422965,0.340428,5861,8703,0.800833,3664,0.663921,0.698440,0.680743,577,607,0.261298,0.457225,0.332549,1391,2434,0.326382,0.492655,0.392641,1906,2877,0.455000,0.298361,0.360396,305,200,0.158607,0.243757,0.192172,1682,2585
3,1.187876,1.091800,0.298509,0.447535,0.358138,5861,8787,0.808276,3636,0.702093,0.755633,0.727880,577,621,0.277027,0.471603,0.349029,1391,2368,0.354518,0.485834,0.409916,1906,2612,0.538462,0.390164,0.452471,305,221,0.163912,0.288942,0.209167,1682,2965
4,1.088721,1.064779,0.310010,0.463402,0.371495,5861,8761,0.812328,3358,0.716323,0.783362,0.748344,577,631,0.284415,0.494608,0.361155,1391,2419,0.365190,0.493179,0.419643,1906,2574,0.533333,0.472131,0.500870,305,270,0.171608,0.292509,0.216311,1682,2867
5,1.036016,1.050908,0.315394,0.472957,0.378430,5861,8789,0.813442,3346,0.740620,0.786828,0.763025,577,613,0.284884,0.493170,0.361148,1391,2408,0.360132,0.514690,0.423758,1906,2724,0.540636,0.501639,0.520408,305,283,0.180369,0.296076,0.224173,1682,2761
6,1.125350,1.042028,0.320319,0.473127,0.382009,5861,8657,0.814427,3230,0.747954,0.792028,0.769360,577,611,0.284879,0.490295,0.360370,1391,2394,0.365644,0.512592,0.426824,1906,2672,0.531987,0.518033,0.524917,305,297,0.185986,0.296671,0.228637,1682,2683
7,1.120945,1.038628,0.314898,0.478929,0.379966,5861,8914,0.814750,3440,0.735341,0.804159,0.768212,577,631,0.283490,0.497484,0.361169,1391,2441,0.371790,0.508919,0.429679,1906,2609,0.545139,0.514754,0.529511,305,288,0.177929,0.311534,0.226497,1682,2945
8,1.058994,1.035759,0.319224,0.477393,0.382606,5861,8765,0.815396,3294,0.741214,0.804159,0.771405,577,626,0.285655,0.492451,0.361573,1391,2398,0.373369,0.510493,0.431294,1906,2606,0.544218,0.524590,0.534224,305,294,0.181626,0.306778,0.228167,1682,2841


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Support,Predicted,Accuracy,Invalid Bio Transitions,Precision Chemical,Recall Chemical,F1 Chemical,Support Chemical,Predicted Chemical,Precision Disease,Recall Disease,F1 Disease,Support Disease,Predicted Disease,Precision Procedure,Recall Procedure,F1 Procedure,Support Procedure,Predicted Procedure,Precision Protein,Recall Protein,F1 Protein,Support Protein,Predicted Protein,Precision Symptom,Recall Symptom,F1 Symptom,Support Symptom,Predicted Symptom
0.547504,0.699757,8,0.581936,0.687084,0.630154,5861,6920,0.886806,706,0.671131,0.781629,0.722178,577,672,0.602526,0.720345,0.656189,1391,1663,0.609502,0.706716,0.654519,1906,2210,0.484848,0.472131,0.478405,305,297,0.521174,0.643876,0.576064,1682,2078


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

#### clinical — Evolución de validación

| Época | Pérdida | Precisión % | Recall % | F1 % |
| --- | --- | --- | --- | --- |
| 1.0 | 1.9194 | 5.03 | 0.26 | 0.49 |
| 2.0 | 0.9572 | 38.65 | 47.11 | 42.46 |
| 3.0 | 0.7877 | 51.32 | 62.24 | 56.26 |
| 4.0 | 0.7378 | 54.14 | 64.65 | 58.93 |
| 5.0 | 0.7092 | 56.87 | 66.85 | 61.45 |
| 6.0 | 0.7026 | 57.4 | 67.79 | 62.17 |
| 7.0 | 0.6947 | 58.19 | 68.4 | 62.88 |
| 8.0 | 0.6998 | 58.19 | 68.71 | 63.02 |
| 8.0 | 0.6998 | 58.19 | 68.71 | 63.02 |

### PASO 09 OK

Finalizaron **4 entrenamientos**. Continúa con el paso 10.

### PASO 10 — Comparación de los dos modelos

Mostramos precisión, recall y F1 por documento, tiempo y mejor época para cada corrida. Se verifica la misma huella del gold y el mismo soporte por categoría entre los dos modelos. Con una semilla se informa el resultado individual y la diferencia RoBERTa − BETO; no se inventa una desviación estándar. Si se amplían las semillas, se calcula media y desviación muestral. Si se añade `frozen`, se calcula también su comparación pareada.


In [19]:
# PASO 10 - Comparación válida para una o varias semillas y variantes
experiments = REPORT.get('experiments', {})
assert set(experiments) == set(CFG['checkpoints']), 'Faltan modelos: ejecuta el paso 09.'
rows, label_rows = [], []
for checkpoint_key in CFG['checkpoints']:
    assert set(experiments[checkpoint_key]) == set(CFG['variants'])
    for variant in CFG['variants']:
        assert set(experiments[checkpoint_key][variant]) == {str(s) for s in CFG['training_seeds']}
        for seed in CFG['training_seeds']:
            result = experiments[checkpoint_key][variant][str(seed)]
            assert result['seed'] == seed and result['split_sha256'] == split_sha256
            assert result['gold_sha256'] == REPORT['common_gold']['validation']['sha256']
            metrics = result['validation']['micro']
            assert metrics['support'] == REPORT['common_gold']['validation']['retained']
            rows.append({'checkpoint': checkpoint_key, 'variante': variant, 'semilla': seed,
                         'learning_rate': result['learning_rate'], 'mejor_epoca': result['best_epoch'],
                         'epocas_ejecutadas': result['epochs_completed'], 'pasos': result['optimizer_steps'],
                         'precision_pct': 100*metrics['precision'], 'recall_pct': 100*metrics['recall'],
                         'f1_pct': 100*metrics['f1'], 'train_segundos': result['train_runtime_seconds']})
            for label in LABELS:
                m = result['validation']['per_label'][label]
                label_rows.append({'checkpoint': checkpoint_key, 'variante': variant, 'semilla': seed,
                                   'categoria': label, 'entidades_validacion': m['support'],
                                   'precision_pct': 100*m['precision'], 'recall_pct': 100*m['recall'],
                                   'f1_pct': 100*m['f1']})
comparison, per_label = pd.DataFrame(rows), pd.DataFrame(label_rows)
assert (per_label.groupby('categoria').entidades_validacion.nunique() == 1).all()
aggregate_rows = []
for (key, variant), group in comparison.groupby(['checkpoint', 'variante'], sort=False):
    row = {'checkpoint': key, 'variante': variant, 'n_semillas': len(group)}
    for metric in ['precision_pct', 'recall_pct', 'f1_pct', 'train_segundos']:
        row[metric + '_media'] = float(group[metric].mean())
        row[metric + '_std'] = float(group[metric].std(ddof=1)) if len(group) > 1 else None
    aggregate_rows.append(row)
aggregate = pd.DataFrame(aggregate_rows)
paired_checkpoint_rows, paired_variant_rows = [], []
for variant in CFG['variants']:
    table = comparison[comparison.variante == variant].pivot(index='semilla', columns='checkpoint', values='f1_pct')
    for seed in CFG['training_seeds']:
        paired_checkpoint_rows.append({'variante': variant, 'semilla': seed,
                                       'f1_beto_pct': float(table.loc[seed, 'beto']),
                                       'f1_clinical_pct': float(table.loc[seed, 'clinical']),
                                       'diferencia_pp': float(table.loc[seed, 'clinical'] - table.loc[seed, 'beto'])})
if {'frozen', 'fine_tuned'} <= set(CFG['variants']):
    for key in CFG['checkpoints']:
        table = comparison[comparison.checkpoint == key].pivot(index='semilla', columns='variante', values='f1_pct')
        for seed in CFG['training_seeds']:
            paired_variant_rows.append({'checkpoint': key, 'semilla': seed,
                                        'diferencia_pp': float(table.loc[seed, 'fine_tuned'] - table.loc[seed, 'frozen'])})
paired_checkpoint = pd.DataFrame(paired_checkpoint_rows)
paired_variant = pd.DataFrame(paired_variant_rows, columns=['checkpoint', 'semilla', 'diferencia_pp'])
for name, table in [('comparison_runs', comparison), ('comparison_aggregate', aggregate),
                    ('comparison_per_label_runs', per_label), ('comparison_paired_checkpoint', paired_checkpoint),
                    ('comparison_paired_variant', paired_variant)]:
    table.to_csv(RUN / f'{name}.csv', index=False)
    if not table.empty:
        shown = table.round(3).astype(object).where(pd.notna(table), None)
        mostrar_tabla(name, list(shown.columns), shown.itertuples(index=False, name=None))
REPORT['comparison'] = {'per_run': rows, 'aggregate': aggregate_rows, 'per_label': label_rows,
                        'paired_checkpoint': paired_checkpoint_rows, 'paired_variant': paired_variant_rows,
                        'note': 'Una semilla: prueba exploratoria; varias: media y desviación muestral, sin prueba de significancia.'}
REPORT['elapsed_seconds_since_cell02'] = time.perf_counter() - START
REPORT['status'] = 'COMPARACION_CONFIGURADA_COMPLETADA'
(RUN / 'summary.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
if len(CFG['training_seeds']) == 1:
    display(Markdown('**Prueba de una semilla:** la desviación estándar no es estimable. '
                     'La diferencia observada entre modelos requiere confirmación con más semillas.'))
display(Markdown(f'### PASO 10 OK\n\nResultados y curvas guardados en `{RUN}`. Test reservado.'))


#### comparison_runs

| checkpoint | variante | semilla | learning_rate | mejor_epoca | epocas_ejecutadas | pasos | precision_pct | recall_pct | f1_pct | train_segundos |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| beto | frozen | 123 | 0.001 | 8.0 | 8.0 | 480 | 31.922 | 47.739 | 38.261 | 215.146 |
| beto | fine_tuned | 123 | 0.0 | 6.0 | 8.0 | 480 | 59.716 | 68.265 | 63.705 | 509.776 |
| clinical | frozen | 123 | 0.001 | 8.0 | 8.0 | 424 | 29.284 | 42.45 | 34.659 | 195.842 |
| clinical | fine_tuned | 123 | 0.0 | 8.0 | 8.0 | 424 | 58.194 | 68.708 | 63.015 | 459.055 |

#### comparison_aggregate

| checkpoint | variante | n_semillas | precision_pct_media | precision_pct_std | recall_pct_media | recall_pct_std | f1_pct_media | f1_pct_std | train_segundos_media | train_segundos_std |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| beto | frozen | 1 | 31.922 | No estimable | 47.739 | No estimable | 38.261 | No estimable | 215.146 | No estimable |
| beto | fine_tuned | 1 | 59.716 | No estimable | 68.265 | No estimable | 63.705 | No estimable | 509.776 | No estimable |
| clinical | frozen | 1 | 29.284 | No estimable | 42.45 | No estimable | 34.659 | No estimable | 195.842 | No estimable |
| clinical | fine_tuned | 1 | 58.194 | No estimable | 68.708 | No estimable | 63.015 | No estimable | 459.055 | No estimable |

#### comparison_per_label_runs

| checkpoint | variante | semilla | categoria | entidades_validacion | precision_pct | recall_pct | f1_pct |
| --- | --- | --- | --- | --- | --- | --- | --- |
| beto | frozen | 123 | CHEMICAL | 577 | 74.121 | 80.416 | 77.14 |
| beto | frozen | 123 | DISEASE | 1391 | 28.565 | 49.245 | 36.157 |
| beto | frozen | 123 | PROCEDURE | 1906 | 37.337 | 51.049 | 43.129 |
| beto | frozen | 123 | PROTEIN | 305 | 54.422 | 52.459 | 53.422 |
| beto | frozen | 123 | SYMPTOM | 1682 | 18.163 | 30.678 | 22.817 |
| beto | fine_tuned | 123 | CHEMICAL | 577 | 82.423 | 83.709 | 83.061 |
| beto | fine_tuned | 123 | DISEASE | 1391 | 58.701 | 68.871 | 63.381 |
| beto | fine_tuned | 123 | PROCEDURE | 1906 | 62.064 | 69.098 | 65.392 |
| beto | fine_tuned | 123 | PROTEIN | 305 | 57.183 | 66.557 | 61.515 |
| beto | fine_tuned | 123 | SYMPTOM | 1682 | 51.87 | 61.831 | 56.414 |
| clinical | frozen | 123 | CHEMICAL | 577 | 69.618 | 69.497 | 69.558 |
| clinical | frozen | 123 | DISEASE | 1391 | 28.739 | 45.723 | 35.294 |
| clinical | frozen | 123 | PROCEDURE | 1906 | 33.294 | 44.753 | 38.183 |
| clinical | frozen | 123 | PROTEIN | 305 | 51.813 | 32.787 | 40.161 |
| clinical | frozen | 123 | SYMPTOM | 1682 | 16.87 | 29.608 | 21.493 |
| clinical | fine_tuned | 123 | CHEMICAL | 577 | 67.113 | 78.163 | 72.218 |
| clinical | fine_tuned | 123 | DISEASE | 1391 | 60.253 | 72.035 | 65.619 |
| clinical | fine_tuned | 123 | PROCEDURE | 1906 | 60.95 | 70.672 | 65.452 |
| clinical | fine_tuned | 123 | PROTEIN | 305 | 48.485 | 47.213 | 47.841 |
| clinical | fine_tuned | 123 | SYMPTOM | 1682 | 52.117 | 64.388 | 57.606 |

#### comparison_paired_checkpoint

| variante | semilla | f1_beto_pct | f1_clinical_pct | diferencia_pp |
| --- | --- | --- | --- | --- |
| frozen | 123 | 38.261 | 34.659 | -3.602 |
| fine_tuned | 123 | 63.705 | 63.015 | -0.69 |

#### comparison_paired_variant

| checkpoint | semilla | diferencia_pp |
| --- | --- | --- |
| beto | 123 | 25.444 |
| clinical | 123 | 28.356 |

**Prueba de una semilla:** la desviación estándar no es estimable. La diferencia observada entre modelos requiere confirmación con más semillas.

### PASO 10 OK

Resultados y curvas guardados en `/kaggle/working/spaccc_transformers/20260926_235331_750165`. Test reservado.

In [20]:

from pathlib import Path

encontrados = []
for carpeta in ['/kaggle/working', '/kaggle/input']:
  raiz = Path(carpeta)
  if raiz.exists():
      for nombre in ['result.json', 'experiments_progress.json', 'summary.json']:
          encontrados.extend(raiz.rglob(nombre))

if encontrados:
  for archivo in sorted(encontrados):
      print(archivo)
else:
  print('No se encontraron resultados en esta sesión.')


/kaggle/working/spaccc_transformers/20260926_235331_750165/beto/fine_tuned/seed_123/result.json
/kaggle/working/spaccc_transformers/20260926_235331_750165/beto/frozen/seed_123/result.json
/kaggle/working/spaccc_transformers/20260926_235331_750165/clinical/fine_tuned/seed_123/result.json
/kaggle/working/spaccc_transformers/20260926_235331_750165/clinical/frozen/seed_123/result.json
/kaggle/working/spaccc_transformers/20260926_235331_750165/experiments_progress.json
/kaggle/working/spaccc_transformers/20260926_235331_750165/summary.json


### PASO 11 — Test opcional, reservado

Desactivado por defecto. Actívalo únicamente después de fijar modelo, variante y semilla, usando las claves `final_test_*` del paso 02. Emplea la misma preparación por palabras y la misma evaluación por documento. No se selecciona automáticamente una combinación ni se usan resultados de test para ajustar la tasa de aprendizaje.


In [24]:
# PASO 11 - Evaluación final de BETO fine_tuned; no realiza entrenamiento
if 'CFG' not in globals() or 'REPORT' not in globals():
    raise RuntimeError('La sesión no conserva CFG/REPORT. Recupera los resultados y el entorno antes de evaluar; '
                       'no basta con ejecutar el paso 02 de nuevo.')
# Selección explícita del usuario tras comparar el F1 de validación.
CFG.update(run_final_test=True, final_test_checkpoint='beto',
           final_test_variant='fine_tuned', final_test_seed=123)
REPORT['config'].update({k: CFG[k] for k in (
    'run_final_test', 'final_test_checkpoint', 'final_test_variant', 'final_test_seed')})
if not CFG['run_final_test']:
    display(Markdown('### PASO 11 OMITIDO\n\nTest reservado; la comparación usa validación.'))
else:
    assert CFG['fraction'] == 1.0
    checkpoint_key, variant, seed = (CFG['final_test_checkpoint'], CFG['final_test_variant'], CFG['final_test_seed'])
    assert checkpoint_key in CFG['checkpoints'] and variant in CFG['variants'] and seed in CFG['training_seeds'], (
        'Selecciona explícitamente final_test_checkpoint, final_test_variant y final_test_seed.')
    result = REPORT.get('experiments', {}).get(checkpoint_key, {}).get(variant, {}).get(str(seed))
    if result is None:
        raise RuntimeError('No están cargados los resultados de BETO fine_tuned, semilla 123. Recupera la corrida del paso 09.')
    if not Path(result['checkpoint_dir']).is_dir():
        raise FileNotFoundError(f"No se encuentra el checkpoint entrenado: {result['checkpoint_dir']}")
    test_documents, test_audit = prepare_common_documents(official_test_ids, df_test)
    test_examples, test_stats = make_examples(checkpoint_key, test_documents, 'test')
    model = load_saved_model(result['checkpoint_dir'])
    test_args = TrainingArguments(output_dir=str(RUN / 'final_test_tmp'),
                                  per_device_eval_batch_size=CFG['batch_size'], report_to=[])
    test_trainer = Trainer(model=model, args=test_args, data_collator=collators[checkpoint_key],
                           processing_class=tokenizers[checkpoint_key],
                           compute_metrics=make_document_metrics(test_examples, test_documents),
                           preprocess_logits_for_metrics=preprocess_logits_for_metrics)
    try:
        test_metrics = test_trainer.evaluate(Dataset.from_list(model_features(test_examples)))
        final_test = {'checkpoint': checkpoint_key, 'variant': variant, 'seed': seed,
                      'metrics': extract_metrics(test_metrics), 'alignment': test_stats, 'gold_audit': test_audit,
                      'metric_scope': REPORT['metric_scope']}
        (RUN / f'final_test_{checkpoint_key}_{variant}_seed_{seed}.json').write_text(
            json.dumps(final_test, indent=2), encoding='utf-8')
        REPORT['final_test'] = final_test
        (RUN / 'summary.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
        mostrar_tabla_configuracion('Test final — BETO fine_tuned, semilla 123', final_test)
    finally:
        del test_trainer, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


Ventanas beto test:   0%|          | 0/250 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Step,Precision,Recall,F1,Support,Predicted,Accuracy,Invalid Bio Transitions,Precision Chemical,Recall Chemical,F1 Chemical,Support Chemical,Predicted Chemical,Precision Disease,Recall Disease,F1 Disease,Support Disease,Predicted Disease,Precision Procedure,Recall Procedure,F1 Procedure,Support Procedure,Predicted Procedure,Precision Protein,Recall Protein,F1 Protein,Support Protein,Predicted Protein,Precision Symptom,Recall Symptom,F1 Symptom,Support Symptom,Predicted Symptom
No log,0.748552,0,0.585791,0.670844,0.625439,9415,10782,0.880488,1205,0.781030,0.737017,0.758385,905,854,0.600294,0.689743,0.641917,2369,2722,0.620143,0.703151,0.659044,2951,3346,0.528861,0.729032,0.613020,465,641,0.497359,0.587523,0.538694,2725,3219


#### Test final — BETO fine_tuned, semilla 123

| Parámetro | Valor |
| --- | --- |
| checkpoint | beto |
| variant | fine_tuned |
| seed | 123 |
| metrics.micro.precision | 0.5857911333704322 |
| metrics.micro.recall | 0.6708443972384492 |
| metrics.micro.f1 | 0.6254394216962915 |
| metrics.micro.support | 9415 |
| metrics.micro.predicted | 10782 |
| metrics.per_label.CHEMICAL.precision | 0.7810304449648712 |
| metrics.per_label.CHEMICAL.recall | 0.7370165745856354 |
| metrics.per_label.CHEMICAL.f1 | 0.7583854462762933 |
| metrics.per_label.CHEMICAL.support | 905 |
| metrics.per_label.DISEASE.precision | 0.6002939015429831 |
| metrics.per_label.DISEASE.recall | 0.6897425073870832 |
| metrics.per_label.DISEASE.f1 | 0.6419171086230603 |
| metrics.per_label.DISEASE.support | 2369 |
| metrics.per_label.PROCEDURE.precision | 0.6201434548714884 |
| metrics.per_label.PROCEDURE.recall | 0.7031514740765842 |
| metrics.per_label.PROCEDURE.f1 | 0.659043989201207 |
| metrics.per_label.PROCEDURE.support | 2951 |
| metrics.per_label.PROTEIN.precision | 0.5288611544461779 |
| metrics.per_label.PROTEIN.recall | 0.7290322580645161 |
| metrics.per_label.PROTEIN.f1 | 0.6130198915009042 |
| metrics.per_label.PROTEIN.support | 465 |
| metrics.per_label.SYMPTOM.precision | 0.4973594283939112 |
| metrics.per_label.SYMPTOM.recall | 0.5875229357798165 |
| metrics.per_label.SYMPTOM.f1 | 0.5386944818304172 |
| metrics.per_label.SYMPTOM.support | 2725 |
| metrics.loss | 0.7485515475273132 |
| metrics.accuracy | 0.8804880014507062 |
| metrics.invalid_bio_transitions | 1205 |
| alignment.documents | 250 |
| alignment.sequences | 399 |
| alignment.supervised_words | 99262 |
| alignment.retained | 9415 |
| alignment.gold_sha256 | 4e20072dd997ee8a52a921850f36f0a368bf8cdea86bb283fceb93ebeed2787f |
| alignment.policy | shared-spacy-first-subtoken-overlap-owner-document-v1 |
| alignment.seconds_this_run | 0.7279238770001939 |
| gold_audit.documents | 250 |
| gold_audit.annotations | 11239 |
| gold_audit.duplicates | 0 |
| gold_audit.unaligned | 136 |
| gold_audit.overlap_excluded | 1688 |
| gold_audit.retained | 9415 |
| gold_audit.words | 99791 |
| gold_audit.ignored_words | 529 |
| metric_scope | F1 micro de entidades exactas por documento y offsets de caracteres, gold común. |

### PASO 12 — Identificación en texto nuevo

Se utiliza el mismo tokenizador de palabras, ventanas, supervisión del primer subtoken y reconstrucción por documento que en entrenamiento. La semilla inicial es 123. Las entidades se presentan con texto, categoría y offsets; sus predicciones no sustituyen una evaluación clínica.


In [26]:
# PASO 12 - Inferencia con la misma preparación por palabras
TEXT_TO_IDENTIFY = ('Paciente con cefalea intensa y fiebre. Se indicó tratamiento '
                    'con paracetamol y se solicitó una resonancia magnética.')
IDENTIFICATION_CHECKPOINT = 'beto'
IDENTIFICATION_VARIANT = 'fine_tuned'
IDENTIFICATION_SEED = 123
assert isinstance(TEXT_TO_IDENTIFY, str) and TEXT_TO_IDENTIFY.strip()
result = REPORT.get('experiments', {}).get(IDENTIFICATION_CHECKPOINT, {}).get(
    IDENTIFICATION_VARIANT, {}).get(str(IDENTIFICATION_SEED))
assert result is not None, 'Ejecuta el paso 09 para la combinación elegida.'
tokens = [t for t in word_nlp(TEXT_TO_IDENTIFY) if not t.is_space]
assert tokens, 'El texto no contiene palabras.'
document = {'doc_id': 'nuevo', 'words': [t.text for t in tokens],
            'offsets': [(t.idx, t.idx + len(t)) for t in tokens],
            'labels': [0] * len(tokens), 'entities': []}
examples = build_word_windows(document, tokenizers[IDENTIFICATION_CHECKPOINT])
model = load_saved_model(result['checkpoint_dir']).to(device)
model.eval()
predictions = []
try:
    with torch.inference_mode():
        for start in range(0, len(examples), CFG['batch_size']):
            batch = collators[IDENTIFICATION_CHECKPOINT](model_features(examples[start:start+CFG['batch_size']]))
            batch.pop('labels')
            batch = {k: v.to(device) for k, v in batch.items()}
            predictions.extend(model(**batch).logits.argmax(dim=-1).cpu().tolist())
    words = reconstruct_word_predictions(predictions, examples, {'nuevo': document})['nuevo']
    spans, invalid = character_entities(words, document['offsets'])
    mostrar_tabla('Entidades detectadas', ['Texto', 'Categoría', 'Inicio', 'Fin'],
                  [[TEXT_TO_IDENTIFY[a:b], label, a, b] for a, b, label in sorted(spans)])
    display(Markdown(f'**Entidades:** {len(spans)}. **Transiciones BIO reparadas:** {invalid}.'))
finally:
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

#### Entidades detectadas

| Texto | Categoría | Inicio | Fin |
| --- | --- | --- | --- |
| cefalea intensa | SYMPTOM | 13 | 28 |
| fiebre | SYMPTOM | 31 | 37 |
| paracetamol | CHEMICAL | 65 | 76 |
| resonancia magnética | PROCEDURE | 95 | 115 |

**Entidades:** 4. **Transiciones BIO reparadas:** 0.

# Informe final — Taller 3: reconocimiento de entidades clínicas con BETO y RoBERTa


## 1. Resultados principales del taller 3

| Modelo | Estrategia | Precisión | Recall | F1 | Mejor época | Épocas ejecutadas | Pasos | Tiempo de entrenamiento |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| BETO | frozen | 31,922 % | 47,739 % | 38,261 % | 8 | 8 | 480 | 215,146 s |
| BETO | fine_tuned | **59,716 %** | 68,265 % | **63,705 %** | 6 | 8 | 480 | 509,776 s |
| RoBERTa clínico | frozen | 29,284 % | 42,450 % | 34,659 % | 8 | 8 | 424 | 195,842 s |
| RoBERTa clínico | fine_tuned | 58,194 % | **68,708 %** | 63,015 % | 8 | 8 | 424 | 459,055 s |

**BETO fine_tuned obtuvo el mayor F1 observado: 63,705 %.** Su mejor checkpoint corresponde a la época 6, aunque la corrida completó 8 épocas. RoBERTa fine_tuned quedó a **0,690 puntos porcentuales**, con un recall ligeramente superior y una precisión inferior. La ventaja global observada de BETO proviene de un mejor equilibrio entre ambas métricas; no implica que supere a RoBERTa en todas las categorías.

Las cuatro corridas alcanzaron el máximo de 8 épocas. Como las variantes frozen y RoBERTa fine_tuned seleccionaron la última época, estos resultados no demuestran que hubieran alcanzado su máximo desempeño posible. BETO fine_tuned sí seleccionó una época anterior al final.

Los modelos ajustados mostraron recall superior a precisión. Esto indica que aún existen falsos positivos y errores en la delimitación de entidades; al exigir coincidencia exacta, un límite incorrecto puede producir simultáneamente una entidad predicha incorrecta y una entidad real no recuperada. Estas métricas no permiten atribuir todos los errores a una causa única.

## 2. Efecto de la estrategia de adaptación

| Comparación con semilla 123 | Diferencia de F1 |
| --- | ---: |
| BETO: fine_tuned − frozen | **+25,444 pp** |
| RoBERTa: fine_tuned − frozen | **+28,356 pp** |
| RoBERTa − BETO, con frozen | −3,602 pp |
| RoBERTa − BETO, con fine_tuned | −0,690 pp |

El cambio de estrategia de adaptación produjo diferencias mucho mayores que el cambio de checkpoint. El ajuste completo mejoró simultáneamente precisión, recall y F1 en ambos modelos. Estos resultados respaldan actualizar las representaciones preentrenadas para esta tarea, en lugar de entrenar únicamente un clasificador sobre representaciones fijas.

La comparación frozen/fine_tuned también incluye tasas de aprendizaje distintas, por lo que no aísla exclusivamente el efecto de congelar el encoder. Dentro de cada estrategia, BETO y RoBERTa comparten la tasa y el protocolo de evaluación.

El ajuste completo requirió aproximadamente **2,37 veces** el tiempo de frozen en BETO y **2,34 veces** en RoBERTa. RoBERTa fine_tuned tardó aproximadamente **51 segundos menos** que BETO fine_tuned, una reducción observada cercana al **10 %**. Sin embargo, también ejecutó menos pasos, 424 frente a 480, porque cada tokenizador genera distintas ventanas. Estos tiempos describen las corridas realizadas y no constituyen una comparación de velocidad con idéntico número de actualizaciones.

## 3. Desempeño por categoría clínica

La siguiente tabla presenta el **F1 por categoría**, con el mismo soporte para los cuatro escenarios:

| Categoría | Entidades de validación | BETO frozen | BETO fine_tuned | RoBERTa frozen | RoBERTa fine_tuned |
| --- | ---: | ---: | ---: | ---: | ---: |
| CHEMICAL | 577 | 77,140 % | **83,061 %** | 69,558 % | 72,218 % |
| DISEASE | 1.391 | 36,157 % | 63,381 % | 35,294 % | **65,619 %** |
| PROCEDURE | 1.906 | 43,129 % | 65,392 % | 38,183 % | **65,452 %** |
| PROTEIN | 305 | 53,422 % | **61,515 %** | 40,161 % | 47,841 % |
| SYMPTOM | 1.682 | 22,817 % | 56,414 % | 21,493 % | **57,606 %** |
| **Total de entidades** | **5.861** | — | — | — | — |

BETO fine_tuned presenta ventajas especialmente claras en **CHEMICAL**, con **10,843 pp** más que RoBERTa fine_tuned, y **PROTEIN**, con **13,674 pp** más. RoBERTa fine_tuned obtiene mejores valores en DISEASE (**+2,238 pp**), SYMPTOM (**+1,192 pp**) y PROCEDURE (**+0,060 pp**). La diferencia en procedimientos es muy pequeña en esta corrida.

El ajuste completo mejora todas las categorías en ambos modelos. En particular, el F1 de SYMPTOM pasa de 22,817 % a 56,414 % en BETO y de 21,493 % a 57,606 % en RoBERTa. No obstante, SYMPTOM sigue siendo la categoría de menor F1 de BETO fine_tuned; en RoBERTa fine_tuned, la menor puntuación corresponde a PROTEIN.

La frecuencia de una categoría no explica por sí sola su dificultad: CHEMICAL tiene menos entidades que enfermedades, procedimientos o síntomas, pero alcanza el mayor F1 de BETO. Para explicar las diferencias entre modelos haría falta inspeccionar falsos positivos, entidades omitidas y errores de límites. El F1 global es **micro**, calculado a partir de conteos agregados, y no el promedio simple de estos cinco F1.

## 4. Comparación de los tres talleres

Los talleres recorren tres enfoques: una Bi-LSTM entrenada desde cero, un Transformer encoder entrenado desde cero y Transformers preentrenados adaptados a NER. La tabla conserva el tipo de estimación de cada experimento para evitar presentar una corrida individual como si fuera una media multisemilla.

| Taller | Modelo y variante | Semillas | Precisión | Recall | F1 de validación |
| --- | --- | --- | ---: | ---: | ---: |
| 1 | Bi-LSTM con POS | 42, 123, 2026 | 32,44 % | 48,76 % | **38,96 % ± 0,51** |
| 1 | Bi-LSTM sin POS | 42, 123, 2026 | 30,90 % | 46,53 % | 37,13 % ± 1,30 |
| 2 | Transformer desde cero con POS | 42, 123, 2026 | 19,94 % | 40,54 % | **26,69 % ± 1,73** |
| 2 | Transformer desde cero sin POS | 42, 123, 2026 | 18,21 % | 40,14 % | 24,98 % ± 2,26 |
| 3 | BETO frozen | 123 | 31,922 % | 47,739 % | 38,261 % |
| 3 | BETO fine_tuned | 123 | 59,716 % | 68,265 % | **63,705 %** |
| 3 | RoBERTa clínico frozen | 123 | 29,284 % | 42,450 % | 34,659 % |
| 3 | RoBERTa clínico fine_tuned | 123 | 58,194 % | 68,708 % | **63,015 %** |

En los talleres 1 y 2, precisión y recall son medias entre semillas; el F1 se expresa como **media ± desviación estándar muestral en puntos porcentuales**. En el taller 3 son resultados de una corrida por escenario, con desviación estándar no estimable.

En el taller 1, incorporar POS mejora el F1 medio de la Bi-LSTM en **1,83 pp**. En el taller 2, el beneficio medio del POS es de **1,71 pp**, pero el Transformer desde cero con POS queda **12,27 pp** por debajo de la Bi-LSTM con POS. Bajo esas configuraciones, cambiar a una arquitectura Transformer sin preentrenamiento no fue suficiente para mejorar el reconocimiento.

El taller 3 presenta el mayor F1 observado. BETO fine_tuned supera numéricamente la media de la mejor variante del taller 1 en **24,745 pp** y la del taller 2 en **37,015 pp**. Estas diferencias son descriptivas: comparan una corrida con promedios multisemilla y protocolos de evaluación parcialmente diferentes.

También puede tomarse la semilla 123 de los talleres anteriores como referencia: Bi-LSTM con POS obtuvo **38,61 %**, Transformer desde cero con POS **28,11 %** y BETO fine_tuned **63,705 %**. Las diferencias son **25,095 pp** y **35,595 pp**, respectivamente. Compartir la semilla no elimina las diferencias de representación, entrenamiento ni evaluación.


## 5. Modelo seleccionado y evaluación final en test

Se seleccionó **BETO fine_tuned, semilla 123**, por obtener el mayor F1 de validación de los cuatro escenarios: **63,705 %**. La selección corresponde al checkpoint de la **época 6**. 


## 6. Limitaciones y conclusión


Aunque los tres talleres comparten corpus y partición, los talleres 1 y 2 utilizan un protocolo común entre sí, mientras el taller 3 reconstruye entidades por documento con ventanas solapadas y deduplicación. La comparación histórica es orientativa; para una comparación plenamente controlada sería necesario evaluar las predicciones de todos los modelos con el mismo conjunto de referencia y procedimiento. Los tiempos tampoco deben extrapolarse a otros entornos ni tratarse como un benchmark de eficiencia.

**El resultado principal del taller 3 es que el ajuste completo supera ampliamente al encoder congelado en ambos checkpoints.** BETO alcanza el mayor F1 observado, **63,705 %**, y se elige para la evaluación final; RoBERTa clínico obtiene un resultado cercano, **63,015 %**, y destaca en enfermedades y síntomas. La especialización clínica del preentrenamiento no garantiza el mejor resultado global en este corpus y configuración. Frente a los dos talleres anteriores, los resultados respaldan continuar con Transformers preentrenados y ajuste completo.
